# Lightsheet movie explorer

Explore one tracked lightsheet movie (e.g. killifish embryos) from the segmentation/tracking pipeline outputs: a nuclear channel (e.g. NLS) plus, optionally, one reporter channel (e.g. β-catenin/BC1, tbxta-GFP). Assign QC and a reporter status (positive/negative) per nucleus, and export a provenance-stamped parquet for downstream notebooks.

Reporter-agnostic: set the channel names in CONFIG. Nothing here assumes which reporter you imaged or what its threshold should be.

**Inputs (pipeline outputs, read-only):** `segments.zarr` (labels, label = `track_id`), `tracks.csv` (centroids per frame), `tracks_fluo.csv` (mean intensity per nucleus per channel), `cell_class_df.csv` (deep vs EVL), `mask_features.csv` (volume, elongation), and MIP movies for display.

**Memory.** Raw frames (~2.4 GB each) are never loaded: every number here comes from the pipeline tables. Each mask frame is one ~2.1 GB compressed chunk, so mask volumes are read only for image panels, one frame at a time, z-slab by z-slab, and cached as 2D label MIPs (~2.7 GB peak). Each slab re-decompresses its whole chunk, so a new frame may take tens of seconds; the first call prints the time. For whole-movie label MIPs, run `scripts/make_label_mips.py` on a workstation (see README).

**How to run.** Edit only the **CONFIG** cell, then run top to bottom. Each section starts with a short note on what it answers and what to look for. Outputs go to `exports/<dataset>/` and `plots/<dataset>/` in this folder; caches to `cache/<dataset>/`.

| § | Section | Main output |
|---|---|---|
| 0 | Libraries, CONFIG, helpers | |
| 1 | Metadata and units check | shapes, chunk sizes, frame → hpf |
| 2 | Load tables | `df_all` (one row per nucleus per frame) |
| 3 | MIP + masks at one timepoint | alignment check |
| 4 | QC: brightness, volume, elongation, deep class | `qc_reason`, `qc_good_frame`, QC panel |
| 5 | Drift / registration check | per-frame median displacement |
| 6 | Track gaps, divisions, eligibility | `track_ok`, track table |
| 7 | Aggregate center (nuclear density): adaptive vs static | `r_um` (continuous), `dist_bin` (200 µm bins) |
| 8 | Per-nucleus parameters | speed, persistence, v toward aggregate, density, depth |
| 9 | Averages over time | median ± IQR per time bin, persistence-window sensitivity |
| 10 | Depth check | intensity vs depth per channel |
| 11 | Reporter channel: distributions and threshold rules | `{reporter}_status`, event-time columns |
| 12 | Positive-call QC gallery | crops: positive / borderline / negative |
| 13 | Parquet builder | nuclei + tracks parquet, params JSON, embryo summary row |
| 14 | Extended QC: track number and length | turnover, survivorship check |
| 15 | Masks with tails | status-coloured masks + track tails |
| 16 | Run log | `run_log.json` |

## 0. Libraries

In [1]:
import json
import sys
import time
import hashlib
import subprocess
import warnings
import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial import cKDTree
from skimage.filters import threshold_otsu
from skimage.segmentation import find_boundaries

## 0. CONFIG — the only cell you should need to edit

Values marked **CONFIRM** are dataset-specific and not yet checked against this movie's metadata. Section 1 prints what it can verify. The example values describe one BC1 + NLS movie (`track_0000_1600_withflow`); replace them with yours.

In [2]:
# ---------------- paths ----------------
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATASET_NAME = "track_0000_1600_withflow"
EMBRYO_ID = "20251019_G1"            # CONFIRM: biological replicate label written into every output row
DATA_DIR = PROJECT_ROOT / "data" / DATASET_NAME

MASK_PATH = DATA_DIR / "segments.zarr"                     # (T, Z, Y, X) labels; label == track_id
TRACKS_PATH = DATA_DIR / "bonus_files" / "tracks.csv"      # or "tracks_smoothed.csv"
FLUO_PATH = DATA_DIR / "tracks_fluo.csv"                   # track_id, mean_fluo, t, channel
CELL_CLASS_PATH = DATA_DIR / "bonus_files" / "cell_class_df.csv"     # None if absent
MASK_FEATURES_PATH = DATA_DIR / "bonus_files" / "mask_features.csv"  # None if absent
# MIP movies for display only (8-bit, compressed: not for quantification).
# Value is a path, or (path, channel_index) for multi-channel czi/tif MIPs.
MIP_PATHS = {                                              # CONFIRM: same view (G1/G2) as the segmentation
    "nls": DATA_DIR / "MIPs" / "20251019_LS_BC1-NLS_G1-NLS.mov",
    "reporter": DATA_DIR / "MIPs" / "20251019_LS_BC1-NLS_G1-BC1.mov",   # keys must match CHANNEL_NAMES values
}
RAW_METADATA_PATH = None   # a raw .czi of this movie; only its header is read (voxel size, frame interval)
REFERENCE_PARQUET = None   # optional: an older parquet of the same movie with t, track_id, volume_um3, elongation, for a cross-check

OUT_DIR = PROJECT_ROOT / "exports" / DATASET_NAME
PLOT_DIR = PROJECT_ROOT / "plots" / DATASET_NAME
CACHE_DIR = PROJECT_ROOT / "cache" / DATASET_NAME   # label MIPs + table cache (gitignored)

# ---------------- units ----------------
SCALE_VEC_ZYX_UM = (3.0, 0.85, 0.85)      # CONFIRM from raw metadata
FRAME_INTERVAL_SEC = 90.0                 # CONFIRM from raw metadata
T0_HPF = None                             # hpf at frame 0; None -> time axis is "hours from frame 0"

# ---------------- channels ----------------
CHANNEL_NAMES = {0: "reporter", 1: "nls"} # CONFIRM (checked against the MIPs in §3): tracks_fluo.csv channel index -> short name
                                          # e.g. {0: "bc1", 1: "nls"} or {0: "tbxta", 1: "nls"}; names become column prefixes
NUCLEAR_CH = "nls"                        # nuclear / control channel
SECOND_CH = "reporter"                    # reporter channel name; None if the movie only has the nuclear channel

# ---------------- QC ----------------
DEEP_ONLY = True
DEEP_CLASS_VALUE = 1                      # CONFIRM: cell_class_df.frame_class value for deep cells
BRIGHT_QC_CH = NUCLEAR_CH                 # channel used for brightness QC
BRIGHT_LOW_Q, BRIGHT_HIGH_Q = 0.05, 0.95  # per-frame bottom/top 5 %
VOL_LOW_Q, VOL_HIGH_Q = 0.005, 0.995      # per-frame volume quantiles
ELONG_MAX = 3.5                           # check the §4 histogram before keeping this
MIN_TRACK_LEN_FRAMES = 10                 # tracks shorter than this are flagged, not dropped
QC_EXAMPLE_T = None                       # frame for the QC panel; None = middle of the movie

# ---------------- aggregate center (from nuclear density; independent of any reporter) ----------------
CENTER_MODE = "adaptive"                  # "adaptive": recomputed every frame, then smoothed; "static": one center from STATIC_CENTER_T_REF
CENTER_SMOOTH_FRAMES = 25                 # rolling median of the adaptive center (frames)
STATIC_CENTER_T_REF = None                # frame for the static center; None = last frame
DENSITY_CENTER_RADIUS_UM = 50             # neighbourhood radius used to find the densest point
CENTER_MIN_N = 30                         # frames with fewer QC-passing nuclei carry the previous center forward
DIST_BIN_UM = 200                         # width of the binned distance (dist_bin); r_um stays continuous

# ---------------- motion and neighbourhood ----------------
DRIFT_CORRECT = False                     # decide after §5
SPEED_HALF_WINDOW = 1                     # speed from path length over t ± this many frames
PERSIST_WINDOWS = (6, 10, 20)             # frames; persistence = net / path over the window
PERSIST_WINDOW_MAIN = 10
DENSITY_RADIUS_UM = 25
DENSITY_POPULATION = "all"                # "all" segmented nuclei or "deep"
DEPTH_SURFACE = "min_z"                   # embryo surface nearest the detection objective: "min_z" (low z) or "max_z"
DEPTH_TILE_UM = 50                        # local surface = shallowest nuclei within each xy tile of this size

# ---------------- averages over time ----------------
TIME_BIN_FRAMES = 10
AVG_GROUP_BY = None                       # None, "dist_bin", or f"{SECOND_CH}_status" (after §11)

# ---------------- second channel ----------------
HIST_LOG10 = True
HIST_N_SEGMENTS = 3                       # histograms per movie segment (thirds by default)
THRESHOLD_MODE = "fixed"                  # "fixed" | "adaptive" | "percentile"
FIXED_THRESHOLD = None                    # no default: set it after looking at the §11a histograms (reporter mean_fluo units)
ADAPTIVE_METHOD = "quantile"              # "quantile" | "otsu" | "median_mad"
ADAPTIVE_QUANTILE = 0.80
ADAPTIVE_MAD_K = 3.0
ADAPTIVE_SMOOTH_FRAMES = 51
PERCENTILE_TOP = 10                       # "top 10 %" per frame
MIN_CONSECUTIVE_FRAMES = 3                # persistence requirement for "positive" (1 = none); applies to every mode
ONSET_MIN_NEG_FRAMES = 3                  # QC-passing negative frames required before an event to call it an observed onset

# ---------------- visualisation ----------------
VIEW_T = None                             # frame for image panels; None = middle of the movie
TAIL_LENGTH_FRAMES = 20
STATUS_LEVEL = "frame"                    # colour by per-frame status or "track" (ever positive)
VIEW_CROP_YX = "auto"                     # (y0, y1, x0, x1) in mask pixels, "auto" (nuclei bounding box), or None (full frame)
MIP_CONTRAST_PCT = (1, 99.8)
GALLERY_T = None                          # frame for the positive-call gallery; None = 3/4 through the movie
GALLERY_N = 6
GALLERY_CROP_UM = 30
BORDERLINE_FRAC = 0.10                    # |value / threshold - 1| below this = borderline

# ---------------- memory ----------------
SLAB_TARGET_GB = 0.5                      # z-slab size when building a label MIP from the mask volume

EXPORT_VERSION = "v001"

## 0. Helpers (theme, logging, caching, memory, image access)

In [ ]:
for _d in (OUT_DIR, PLOT_DIR, CACHE_DIR, CACHE_DIR / "label_mips"):
    _d.mkdir(parents=True, exist_ok=True)

scale_vec = np.asarray(SCALE_VEC_ZYX_UM, dtype=float)
DT_MIN = FRAME_INTERVAL_SEC / 60.0
TIME_LABEL = "hpf" if T0_HPF is not None else "hours from frame 0"

# ---- theme and palette ----
def apply_dark_theme(fig, font_size=18):
    fig.update_layout(template="plotly_dark", font=dict(size=font_size, color="white"),
                      paper_bgcolor="black", plot_bgcolor="black",
                      legend=dict(bgcolor="rgba(0,0,0,0)", bordercolor="rgba(0,0,0,0)"))
    fig.update_xaxes(showgrid=True, gridcolor="rgba(255,255,255,0.15)", zerolinecolor="rgba(255,255,255,0.3)")
    fig.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.15)", zerolinecolor="rgba(255,255,255,0.3)")
    return fig

STATUS_COLORS = {"positive": "#ffe138", "negative": "#904ed3", "qc_fail": "#7f7f7f", "none": "#4fc3c8"}
SEGMENT_COLORS = ["#6fa8dc", "#f6b26b", "#e06666", "#93c47d", "#c27ba0"]
plt.style.use("dark_background")

def save_fig(fig, name):
    """Save a matplotlib or plotly figure to PLOT_DIR."""
    path = PLOT_DIR / name
    if hasattr(fig, "savefig"):
        fig.savefig(path.with_suffix(".png"), dpi=160, bbox_inches="tight")
    else:
        try:
            fig.write_html(path.with_suffix(".html"))
        except Exception as exc:
            print("could not save", name, exc)
    return path

def frame_to_hours(t):
    return (T0_HPF or 0.0) + np.asarray(t, dtype=float) * FRAME_INTERVAL_SEC / 3600.0

# ---- run log (#11) ----
RUN_LOG = []
def _jsonable(v):
    if isinstance(v, (np.integer,)): return int(v)
    if isinstance(v, (np.floating,)): return None if not np.isfinite(v) else float(v)
    if isinstance(v, float) and not np.isfinite(v): return None
    if isinstance(v, (np.ndarray, tuple, list)): return [_jsonable(x) for x in v]
    if isinstance(v, dict): return {str(k): _jsonable(x) for k, x in v.items()}
    if isinstance(v, Path): return str(v)
    if isinstance(v, (str, int, float, bool)) or v is None: return v
    return str(v)

def log_step(section, **info):
    RUN_LOG.append({"section": section, "time": dt.datetime.now().isoformat(timespec="seconds"),
                    **{k: _jsonable(v) for k, v in info.items()}})

def config_snapshot():
    """All UPPER_CASE settings in this notebook, JSON-safe."""
    keep = (str, int, float, bool, tuple, list, dict, Path, type(None), np.integer, np.floating)
    return {k: _jsonable(v) for k, v in globals().items()
            if k.isupper() and not k.startswith("_") and isinstance(v, keep)}

# ---- caching (#12) ----
def file_fingerprint(p):
    if p is None: return None
    p = Path(p)
    if not p.exists(): return {"path": str(p), "exists": False}
    if p.is_dir():                         # zarr: fingerprint its metadata file
        meta = next((p / m for m in (".zarray", "zarr.json") if (p / m).exists()), None)
        if meta is not None:
            st = meta.stat()
            return {"path": str(p), "metadata_file": meta.name, "bytes": st.st_size, "mtime": int(st.st_mtime),
                    "metadata_sha1": hashlib.sha1(meta.read_bytes()).hexdigest()[:12]}
    st = p.stat()
    return {"path": str(p), "bytes": st.st_size, "mtime": int(st.st_mtime)}

def cache_key(*parts):
    return hashlib.sha1(json.dumps(parts, sort_keys=True, default=str).encode()).hexdigest()[:10]

def write_table(df, path, metadata=None):
    """Parquet with optional JSON metadata in the schema; pickle if pyarrow is missing."""
    path = Path(path)
    try:
        import pyarrow as pa, pyarrow.parquet as pq
        table = pa.Table.from_pandas(df, preserve_index=False)
        if metadata is not None:
            md = dict(table.schema.metadata or {})
            md[b"provenance"] = json.dumps(_jsonable(metadata)).encode()
            table = table.replace_schema_metadata(md)
        pq.write_table(table, path)
        return path
    except ImportError:
        alt = path.with_suffix(".pkl")
        warnings.warn(f"pyarrow not installed; wrote {alt.name} instead of parquet")
        df.to_pickle(alt)
        return alt

def read_table(path):
    path = Path(path)
    if path.exists() and path.suffix != ".pkl":
        return pd.read_parquet(path)
    return pd.read_pickle(path.with_suffix(".pkl"))

def read_csv_fast(path, usecols=None, dtype=None):
    try:
        return pd.read_csv(path, usecols=usecols, dtype=dtype, engine="pyarrow")
    except Exception:
        return pd.read_csv(path, usecols=usecols, dtype=dtype)

# ---- memory guard (tools/ram_budget.py) ----
sys.path.insert(0, str(PROJECT_ROOT / "tools"))
try:
    from ram_budget import check_fits, available_gb, SOFT_GB, HARD_GB
except Exception:
    SOFT_GB, HARD_GB = 4.0, 6.0
    def available_gb(): return None
    def check_fits(shape, dtype, copies=1.0, label=""):
        peak = float(np.prod(shape)) * np.dtype(dtype).itemsize * copies / 1024**3
        if peak > HARD_GB: raise MemoryError(f"[{label}] estimated {peak:.2f} GB > {HARD_GB} GB")
        if peak > SOFT_GB: print(f"WARNING [{label}] estimated {peak:.2f} GB > {SOFT_GB} GB soft limit")
        return {"peak_gb": round(peak, 3)}

# ---- masks ----
def open_label_array(path):
    path = Path(path)
    if path.suffix == ".npy":
        return np.load(path, mmap_mode="r")
    import zarr
    return zarr.open(str(path), mode="r")

def label_mip(t, verbose=True):
    """2D max projection of the label volume at frame t (labels == track_id).

    Order: cached npz (from this notebook or scripts/make_label_mips.py) -> read the
    volume z-slab by z-slab. A partial read still decompresses the whole zarr chunk,
    so peak RAM ~ one chunk + one slab.
    """
    f = CACHE_DIR / "label_mips" / f"t{int(t):05d}.npz"
    if f.exists():
        return np.load(f)["mip"]
    arr = mask_zarr
    _, Z, Y, X = arr.shape
    itemsize = np.dtype(arr.dtype).itemsize
    chunks = getattr(arr, "chunks", None) or (1, Z, Y, X)
    chunk_bytes = int(np.prod(chunks[1:])) * itemsize
    slab = int(max(1, min(chunks[1], SLAB_TARGET_GB * 1024**3 // (Y * X * itemsize))))
    peak = check_fits((chunk_bytes // itemsize + slab * Y * X,), arr.dtype, copies=1.0, label=f"label MIP t={t}")
    if verbose:
        print(f"building label MIP t={t}: z-slabs of {slab}, est. peak {peak['peak_gb']} GB (cached afterwards)")
    _t0 = time.time()
    mip = np.zeros((Y, X), dtype=arr.dtype)
    for z0 in range(0, Z, slab):
        block = np.asarray(arr[int(t), z0:z0 + slab])
        np.maximum(mip, block.max(axis=0), out=mip)
        del block
    np.savez_compressed(f, mip=mip)
    if verbose:
        print(f"  done in {time.time() - _t0:.1f} s")
    return mip

# ---- MIP movies (display only) ----
_MIP_READERS = {}
def _mip_spec(name):
    spec = MIP_PATHS[name]
    return (Path(spec[0]), int(spec[1])) if isinstance(spec, (tuple, list)) else (Path(spec), 0)

def read_mip_frame(name, t):
    """One 2D MIP frame as float32. Never reads the whole movie."""
    path, ch = _mip_spec(name)
    suf = path.suffix.lower()
    if suf in (".mov", ".mp4", ".avi", ".mkv"):
        import imageio.v2 as iio2
        if path not in _MIP_READERS:
            _MIP_READERS[path] = iio2.get_reader(str(path), "ffmpeg")
        fr = _MIP_READERS[path].get_data(int(t))
    elif suf in (".tif", ".tiff"):
        import tifffile
        with tifffile.TiffFile(path) as tf:
            fr = tf.pages[int(t)].asarray()
    elif suf == ".czi":
        from aicsimageio import AICSImage
        if path not in _MIP_READERS:
            _MIP_READERS[path] = AICSImage(str(path))
        fr = _MIP_READERS[path].get_image_dask_data("YX", T=int(t), C=ch, Z=0).compute()
    elif suf == ".npy":
        fr = np.load(path, mmap_mode="r")[int(t)]
    else:
        raise ValueError(f"unsupported MIP format: {path}")
    fr = np.asarray(fr)
    if fr.ndim == 3:                      # RGB video frame -> grey
        fr = fr[..., :3].mean(axis=-1)
    return fr.astype(np.float32)

def mip_frame_count(name):
    path, ch = _mip_spec(name)
    suf = path.suffix.lower()
    try:
        if suf in (".mov", ".mp4", ".avi", ".mkv"):
            import imageio.v2 as iio2
            rdr = _MIP_READERS.setdefault(path, iio2.get_reader(str(path), "ffmpeg"))
            try:
                return int(rdr.count_frames())
            except Exception:
                return int(rdr.get_meta_data().get("nframes"))
        if suf in (".tif", ".tiff"):
            import tifffile
            with tifffile.TiffFile(path) as tf:
                return len(tf.pages)
        if suf == ".czi":
            from aicsimageio import AICSImage
            return int(_MIP_READERS.setdefault(path, AICSImage(str(path))).dims.T)
        if suf == ".npy":
            return int(np.load(path, mmap_mode="r").shape[0])
    except Exception as exc:
        print(f"could not count frames for {name}: {exc}")
    return None

def contrast_limits(img, pct=MIP_CONTRAST_PCT):
    lo, hi = np.percentile(img, pct)
    return float(lo), float(max(hi, lo + 1e-6))

def mip_on_mask_grid_extent():
    """imshow extent that stretches any MIP onto the mask (x, y) pixel grid."""
    return (0, X_MASK, Y_MASK, 0)

def crop_view(ax, crop=None, t=None):
    if isinstance(crop, str) and crop == "auto":
        sub = df_all[df_all["t"] == t] if t is not None else df_all
        m = 40
        crop = (max(sub["cy_px"].min() - m, 0), min(sub["cy_px"].max() + m, Y_MASK),
                max(sub["cx_px"].min() - m, 0), min(sub["cx_px"].max() + m, X_MASK))
    if crop is not None:
        y0, y1, x0, x1 = crop
        ax.set_xlim(x0, x1); ax.set_ylim(y1, y0)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("time axis:", TIME_LABEL, "| frame interval:", FRAME_INTERVAL_SEC, "s")

## 1. Metadata and units check

**Question:** do the masks, tables and MIP movies describe the same movie, in the units we think they do?

Look for: number of frames equal across masks, tables and MIPs; MIP aspect ratio equal to the mask's; voxel size and frame interval confirmed from the raw file (otherwise they stay marked CONFIRM). Speed, persistence and distance are only as good as these numbers.

In [ ]:
mask_zarr = open_label_array(MASK_PATH)
T_MASK, Z_MASK, Y_MASK, X_MASK = mask_zarr.shape
_chunks = getattr(mask_zarr, "chunks", None)
_itemsize = np.dtype(mask_zarr.dtype).itemsize
meta_rows = [
    ("mask shape (T, Z, Y, X)", mask_zarr.shape),
    ("mask dtype", str(mask_zarr.dtype)),
    ("mask chunks", _chunks),
    ("GB per mask frame (uncompressed)", round(Z_MASK * Y_MASK * X_MASK * _itemsize / 1024**3, 2)),
    ("GB decompressed per chunk read", None if _chunks is None else round(np.prod(_chunks) * _itemsize / 1024**3, 2)),
    ("voxel size z, y, x (µm)", tuple(scale_vec)),
    ("field of view y, x (µm)", (round(Y_MASK * scale_vec[1], 1), round(X_MASK * scale_vec[2], 1))),
    ("z extent (µm)", round(Z_MASK * scale_vec[0], 1)),
    ("frame interval (s)", FRAME_INTERVAL_SEC),
    (f"frame 0 -> {TIME_LABEL}", float(frame_to_hours(0))),
    (f"last frame -> {TIME_LABEL}", round(float(frame_to_hours(T_MASK - 1)), 2)),
    ("RAM available now (GB)", available_gb()),
]
mip_info = {}
for name in MIP_PATHS:
    try:
        fr0 = read_mip_frame(name, 0)
        n = mip_frame_count(name)
        mip_info[name] = {"shape": fr0.shape, "n_frames": n,
                          "scale_y": fr0.shape[0] / Y_MASK, "scale_x": fr0.shape[1] / X_MASK}
        meta_rows.append((f"MIP '{name}' frame shape / n frames", (fr0.shape, n)))
    except Exception as exc:
        mip_info[name] = {"error": str(exc)}
        meta_rows.append((f"MIP '{name}'", f"ERROR {exc}"))
display(pd.DataFrame(meta_rows, columns=["item", "value"]).set_index("item"))

for name, info in mip_info.items():
    if "error" in info:
        continue
    if abs(info["scale_y"] - info["scale_x"]) > 0.01:
        print(f"WARNING: MIP '{name}' aspect ratio differs from the mask grid "
              f"(scale y {info['scale_y']:.3f} vs x {info['scale_x']:.3f}); overlays will be distorted or offset.")
    if info["n_frames"] is not None and info["n_frames"] != T_MASK:
        print(f"WARNING: MIP '{name}' has {info['n_frames']} frames but masks have {T_MASK}; "
              "frame indices may not correspond.")

if RAW_METADATA_PATH is not None:
    # Header only: AICSImage does not read pixel data here.
    import re
    from aicsimageio import AICSImage
    raw = AICSImage(str(RAW_METADATA_PATH))
    print("raw dims:", raw.dims, "| physical pixel sizes (Z, Y, X µm):", raw.physical_pixel_sizes)
    pps = raw.physical_pixel_sizes
    if all(v is not None for v in (pps.Z, pps.Y, pps.X)):
        if not np.allclose([pps.Z, pps.Y, pps.X], scale_vec, rtol=0.02):
            print("WARNING: SCALE_VEC_ZYX_UM does not match the raw metadata. Masks may be "
                  "downsampled relative to raw; check before trusting µm values.")
    xml = str(raw.metadata)
    hits = sorted(set(re.findall(r"(?:TimeIncrement|Interval|TimeSpan)[^<>]{0,80}", xml)))[:10]
    print("time-interval fields found in raw metadata (confirm FRAME_INTERVAL_SEC):", hits or "none")
else:
    print("RAW_METADATA_PATH not set: voxel size and frame interval remain UNCONFIRMED "
          "(values typed into CONFIG).")

log_step("1_metadata", mask_shape=mask_zarr.shape, chunks=_chunks, scale_vec=scale_vec,
         frame_interval_sec=FRAME_INTERVAL_SEC, t0_hpf=T0_HPF, mip_info=mip_info,
         raw_metadata_checked=RAW_METADATA_PATH is not None)

## 2. Load tables into `df_all`

One row per nucleus per frame, keyed by (`t`, `track_id`). Column names: (`cz_px`, `cy_px`, `cx_px`, `volume_um3`, `elongation`, `frame_class`). Intensities become one column per channel (`{name}_mean_fluo`, e.g. `reporter_mean_fluo`, `nls_mean_fluo`), each on its own scale.

`volume_um3` is `mask_features.volume`, which is already in µm³ (checked for one pipeline version: label 1 at t=0 had 483.35 = 223 voxels × 2.1675 µm³; check again for yours). `elongation` here is `1 / elongation_ratio` from the pipeline's `mask_features.csv`. That is not the same formula as `sqrt(λmax/λmin)` of the inertia tensor; the optional cross-check in §4 compares the two.

The merged table is cached in `CACHE_DIR`; it is rebuilt automatically if any input file changes.

In [ ]:
_inputs = [TRACKS_PATH, FLUO_PATH, CELL_CLASS_PATH, MASK_FEATURES_PATH]
_key = cache_key([file_fingerprint(p) for p in _inputs], CHANNEL_NAMES, SCALE_VEC_ZYX_UM)
_base_cache = CACHE_DIR / f"base_{_key}.parquet"

def load_base_table():
    tracks = read_csv_fast(TRACKS_PATH)
    tracks = tracks[["track_id", "t", "z", "y", "x"] + (["parent_track_id"] if "parent_track_id" in tracks else [])]
    tracks = tracks.rename(columns={"z": "cz_px", "y": "cy_px", "x": "cx_px"})
    if "parent_track_id" not in tracks:
        tracks["parent_track_id"] = -1
    df = tracks.astype({"track_id": np.int64, "t": np.int32, "cz_px": np.float32,
                        "cy_px": np.float32, "cx_px": np.float32, "parent_track_id": np.int64})
    n_dup = df.duplicated(["t", "track_id"]).sum()
    if n_dup:
        raise ValueError(f"{n_dup} duplicated (t, track_id) rows in {TRACKS_PATH.name}")

    # intensities: long -> wide, one column per channel
    fl = read_csv_fast(FLUO_PATH, usecols=["track_id", "mean_fluo", "t", "channel"])
    n_dup_fl = fl.duplicated(["t", "track_id", "channel"]).sum()
    if n_dup_fl:
        print(f"note: {n_dup_fl} duplicated (t, track_id, channel) intensity rows; averaged")
    fl_wide = fl.pivot_table(index=["t", "track_id"], columns="channel", values="mean_fluo", aggfunc="mean")
    fl_wide.columns = [f"{CHANNEL_NAMES.get(c, f'ch{c}')}_mean_fluo" for c in fl_wide.columns]
    fl_wide = fl_wide.astype(np.float32).reset_index()
    df = df.merge(fl_wide, on=["t", "track_id"], how="left", validate="one_to_one")
    del fl, fl_wide

    if CELL_CLASS_PATH is not None and Path(CELL_CLASS_PATH).exists():
        cc = read_csv_fast(CELL_CLASS_PATH, usecols=["t", "track_id", "frame_class"])
        cc = cc.drop_duplicates(["t", "track_id"])
        df = df.merge(cc, on=["t", "track_id"], how="left", validate="one_to_one")
    else:
        df["frame_class"] = np.nan

    if MASK_FEATURES_PATH is not None and Path(MASK_FEATURES_PATH).exists():
        mf = read_csv_fast(MASK_FEATURES_PATH,
                           usecols=["label", "frame", "volume", "solidity", "elongation_ratio", "bbox_z"])
        mf = mf.rename(columns={"label": "track_id", "frame": "t", "volume": "volume_um3"})
        mf = mf.drop_duplicates(["t", "track_id"])
        mf["elongation"] = 1.0 / mf["elongation_ratio"].where(mf["elongation_ratio"] > 0)
        df = df.merge(mf.astype({"volume_um3": np.float32, "solidity": np.float32,
                                 "elongation_ratio": np.float32, "elongation": np.float32}),
                      on=["t", "track_id"], how="left", validate="one_to_one")
    else:
        for c in ("volume_um3", "solidity", "elongation_ratio", "elongation", "bbox_z"):
            df[c] = np.nan
    return df.sort_values(["t", "track_id"]).reset_index(drop=True)

if _base_cache.exists() or _base_cache.with_suffix(".pkl").exists():
    df_all = read_table(_base_cache)
    print("loaded cached base table:", _base_cache.name)
else:
    df_all = load_base_table()
    write_table(df_all, _base_cache)
    print("built base table and cached it:", _base_cache.name)

# physical coordinates and time
df_all["z_um"] = (df_all["cz_px"] * scale_vec[0]).astype(np.float32)
df_all["y_um"] = (df_all["cy_px"] * scale_vec[1]).astype(np.float32)
df_all["x_um"] = (df_all["cx_px"] * scale_vec[2]).astype(np.float32)
df_all["hours"] = frame_to_hours(df_all["t"]).astype(np.float32)
df_all["embryo_id"] = EMBRYO_ID
FLUO_COLS = [c for c in df_all.columns if c.endswith("_mean_fluo")]

# consistency checks
checks = {
    "rows": len(df_all),
    "frames in tables": df_all["t"].nunique(),
    "t range": (int(df_all["t"].min()), int(df_all["t"].max())),
    "tracks": df_all["track_id"].nunique(),
    "t max < mask T": bool(df_all["t"].max() < T_MASK),
    "centroids inside mask grid": bool((df_all["cz_px"].between(0, Z_MASK) & df_all["cy_px"].between(0, Y_MASK)
                                        & df_all["cx_px"].between(0, X_MASK)).all()),
    **{f"{c} matched": f"{df_all[c].notna().mean():.1%}" for c in FLUO_COLS},
    "frame_class matched": f"{df_all['frame_class'].notna().mean():.1%}",
    "mask features matched": f"{df_all['volume_um3'].notna().mean():.1%}",
    "df_all memory (GB)": round(df_all.memory_usage(deep=True).sum() / 1024**3, 2),
}
display(pd.Series(checks, name="value").to_frame())
if SECOND_CH is not None and f"{SECOND_CH}_mean_fluo" not in df_all:
    print(f"WARNING: no '{SECOND_CH}_mean_fluo' column; second-channel sections will be skipped.")
    SECOND_CH = None
CH2_COL = f"{SECOND_CH}_mean_fluo" if SECOND_CH else None
_t_lo, _t_hi = int(df_all["t"].min()), int(df_all["t"].max())
VIEW_T = int((_t_lo + _t_hi) // 2) if VIEW_T is None else VIEW_T
QC_EXAMPLE_T = VIEW_T if QC_EXAMPLE_T is None else QC_EXAMPLE_T
GALLERY_T = int(_t_lo + 0.75 * (_t_hi - _t_lo)) if GALLERY_T is None else GALLERY_T
STATIC_CENTER_T_REF = _t_hi if STATIC_CENTER_T_REF is None else STATIC_CENTER_T_REF
print(f"frames used for panels: VIEW_T={VIEW_T}, QC_EXAMPLE_T={QC_EXAMPLE_T}, GALLERY_T={GALLERY_T}")
NUC_COL = f"{NUCLEAR_CH}_mean_fluo"
log_step("2_load", **checks)
df_all.head()

## 3. MIP and masks at one timepoint

**Question:** do masks and centroids land on nuclei in the MIP? This is the alignment check for every overlay below.

Left: nuclear MIP. Middle: second-channel MIP. Right: nuclear MIP with label-MIP outlines and table centroids (dots). Dots that consistently sit beside nuclei mean the MIP is offset, flipped or from a different view than the segmentation.

Caveats: the label MIP shows the top-most-numbered label along z, so overlapping nuclei hide each other; MIP movies are 8-bit compressed and for display only.

In [ ]:
def plot_mip_and_masks(t, crop=VIEW_CROP_YX):
    mip_lab = label_mip(t)
    names = [NUCLEAR_CH] + ([SECOND_CH] if SECOND_CH in MIP_PATHS else [])
    imgs = {n: read_mip_frame(n, t) for n in names if n in MIP_PATHS}
    fig, axes = plt.subplots(1, len(imgs) + 1, figsize=(6 * (len(imgs) + 1), 6))
    for ax, (n, img) in zip(axes, imgs.items()):
        ax.imshow(img, cmap="gray", clim=contrast_limits(img), extent=mip_on_mask_grid_extent())
        ax.set_title(f"{n} MIP, t={t}")
    ax = axes[-1]
    base = imgs.get(NUCLEAR_CH, next(iter(imgs.values())))
    ax.imshow(base, cmap="gray", clim=contrast_limits(base), extent=mip_on_mask_grid_extent())
    bnd = find_boundaries(mip_lab, mode="inner")
    ax.imshow(np.ma.masked_where(~bnd, bnd), cmap="autumn", alpha=0.9, interpolation="nearest")
    sub = df_all[df_all["t"] == t]
    ax.scatter(sub["cx_px"], sub["cy_px"], s=3, c="cyan", linewidths=0)
    ax.set_title(f"outlines + centroids (n={len(sub)})")
    for a in axes:
        crop_view(a, crop, t); a.set_xticks([]); a.set_yticks([])
    fig.tight_layout()
    return fig

fig = plot_mip_and_masks(VIEW_T)
save_fig(fig, f"03_mip_masks_t{VIEW_T}")
plt.show()

# Channel identity check: each MIP movie should correlate best with its own tracks_fluo channel.
def check_channel_identity(t=VIEW_T):
    sub = df_all[df_all["t"] == t].dropna(subset=FLUO_COLS)
    res = {}
    for name in MIP_PATHS:
        img = read_mip_frame(name, t)
        sy, sx = img.shape[0] / Y_MASK, img.shape[1] / X_MASK
        yy = np.clip((sub["cy_px"] * sy).astype(int), 0, img.shape[0] - 1)
        xx = np.clip((sub["cx_px"] * sx).astype(int), 0, img.shape[1] - 1)
        pix = pd.Series(img[yy, xx], index=sub.index)
        res[name] = {c: pix.corr(sub[c], method="spearman") for c in FLUO_COLS}
    tab = pd.DataFrame(res).T.round(2)
    print("Spearman correlation: MIP pixel at centroid (rows) vs tracks_fluo channel (columns)")
    display(tab)
    for name in tab.index:
        own = f"{name}_mean_fluo"
        if own in tab.columns and tab.loc[name].idxmax() != own and tab.loc[name].max() - tab.loc[name, own] > 0.1:
            raise ValueError(f"MIP '{name}' matches {tab.loc[name].idxmax()} better than {own}: "
                             "CHANNEL_NAMES or MIP_PATHS are probably swapped.")
    return tab

channel_check = check_channel_identity()
log_step("3_channel_check", table=channel_check.to_dict())

## 4. QC

Flags are added as columns; **no rows are dropped**. `qc_reason` lists every reason a nucleus fails (`;`-separated); `qc_good_frame` is True only if none apply. Reasons:

- `not_deep`: `frame_class != DEEP_CLASS_VALUE` (only when `DEEP_ONLY`)
- `dim` / `bright`: below/above the per-frame `BRIGHT_LOW_Q` / `BRIGHT_HIGH_Q` quantile of `BRIGHT_QC_CH` among deep nuclei. By construction this removes ~10 % of deep nuclei in every frame, including real extremes; it targets segmentation fragments and merges, not biology. Because the nuclear channel dims with depth and changes through the cell cycle, this flag preferentially removes deep nuclei and nuclei around division; §10 shows whether that matters for this movie.
- `volume`: outside per-frame `VOL_LOW_Q` / `VOL_HIGH_Q`
- `elongated`: `elongation > ELONG_MAX`
- `missing`: no intensity or mask features for this row

Track length is handled separately in §6 (`track_ok`).

In [ ]:
def add_qc_flags(df):
    df = df.copy()
    deep = (df["frame_class"] == DEEP_CLASS_VALUE) if DEEP_ONLY else pd.Series(True, index=df.index)
    ref = df[deep]
    qb = ref.groupby("t")[f"{BRIGHT_QC_CH}_mean_fluo"].quantile([BRIGHT_LOW_Q, BRIGHT_HIGH_Q]).unstack()
    qb.columns = ["qc_bright_lo", "qc_bright_hi"]
    qv = ref.groupby("t")["volume_um3"].quantile([VOL_LOW_Q, VOL_HIGH_Q]).unstack()
    qv.columns = ["qc_vol_lo", "qc_vol_hi"]
    df = df.drop(columns=[c for c in ("qc_bright_lo", "qc_bright_hi", "qc_vol_lo", "qc_vol_hi") if c in df])
    df = df.join(qb, on="t").join(qv, on="t")
    b = df[f"{BRIGHT_QC_CH}_mean_fluo"]
    flags = {
        "not_deep": ~deep,
        "missing": df[FLUO_COLS + ["volume_um3"]].isna().any(axis=1),
        "dim": b < df["qc_bright_lo"],
        "bright": b > df["qc_bright_hi"],
        "volume": (df["volume_um3"] < df["qc_vol_lo"]) | (df["volume_um3"] > df["qc_vol_hi"]),
        "elongated": df["elongation"] > ELONG_MAX,
    }
    for k, v in flags.items():
        df[f"qc_bad_{k}"] = v.fillna(False).to_numpy(bool)
    bad = np.column_stack([df[f"qc_bad_{k}"].to_numpy() for k in flags])
    names = np.array(list(flags))
    df["qc_n_fail"] = bad.sum(axis=1).astype(np.int8)
    code = bad.astype(np.int64) @ (1 << np.arange(len(names)))
    lookup = {c: (";".join(names[[(c >> i) & 1 == 1 for i in range(len(names))]]) or "pass") for c in np.unique(code)}
    df["qc_reason"] = pd.Series(code, index=df.index).map(lookup)
    df["qc_primary"] = np.where(bad.any(axis=1), names[bad.argmax(axis=1)], "pass")
    df["qc_good_frame"] = ~bad.any(axis=1)
    return df

df_all = add_qc_flags(df_all)
QC_REASONS = ["not_deep", "missing", "dim", "bright", "volume", "elongated"]

# QC table: share of nuclei failing each reason, overall and per movie segment
_seg_edges = np.linspace(df_all["t"].min(), df_all["t"].max() + 1, HIST_N_SEGMENTS + 1)
df_all["movie_segment"] = pd.cut(df_all["t"], _seg_edges, right=False,
                                 labels=[f"seg{i+1}" for i in range(HIST_N_SEGMENTS)]).astype(str)
qc_table = pd.concat({
    "all": df_all[[f"qc_bad_{r}" for r in QC_REASONS] + ["qc_good_frame"]].mean(),
    **{s: g[[f"qc_bad_{r}" for r in QC_REASONS] + ["qc_good_frame"]].mean()
       for s, g in df_all.groupby("movie_segment")},
}, axis=1).mul(100).round(1)
qc_table.index = [i.replace("qc_bad_", "") for i in qc_table.index]
print("% of all nuclei failing each reason (reasons overlap; 'qc_good_frame' = % passing everything)")
display(qc_table)
deep_only = df_all[~df_all["qc_bad_not_deep"]]
print(f"among deep nuclei: {deep_only['qc_good_frame'].mean():.1%} pass")
qc_table.to_csv(OUT_DIR / "qc_table.csv")
log_step("4_qc", qc_table=qc_table.to_dict(), deep_pass_frac=deep_only["qc_good_frame"].mean())

# per-frame pass fraction among deep nuclei
_pf = deep_only.groupby("t")["qc_good_frame"].mean()
fig = px.line(x=frame_to_hours(_pf.index), y=_pf.values,
              labels={"x": TIME_LABEL, "y": "fraction of deep nuclei passing QC"},
              title="QC pass fraction over time (deep nuclei)")
apply_dark_theme(fig, 14).show()

### QC panel for one frame

Outlines (visible nuclei) and dots (all nuclei, including those hidden in the MIP) are coloured by the first failing reason. Histograms show the cut lines for this frame. Check that `dim`/`bright`/`volume`/`elongated` nuclei look like fragments, merges or debris rather than ordinary nuclei.

In [ ]:
QC_COLORS = {"pass": "#d9d9d9", "not_deep": "#4a86e8", "missing": "#ffffff", "dim": "#8e7cc3",
             "bright": "#ffd966", "volume": "#e06666", "elongated": "#93c47d"}

def plot_qc_panel(t, crop=VIEW_CROP_YX):
    sub = df_all[df_all["t"] == t]
    mip_lab = label_mip(t)
    img = read_mip_frame(NUCLEAR_CH, t)
    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(3, 2, width_ratios=[2.2, 1])
    ax = fig.add_subplot(gs[:, 0])
    ax.imshow(img, cmap="gray", clim=contrast_limits(img), extent=mip_on_mask_grid_extent())
    # outline colour by primary reason
    reason_code = {r: i + 1 for i, r in enumerate(QC_COLORS)}
    lut = np.zeros(int(max(mip_lab.max(), sub["track_id"].max())) + 1, dtype=np.int16)
    lut[sub["track_id"].to_numpy()] = sub["qc_primary"].map(reason_code).to_numpy()
    bnd = find_boundaries(mip_lab, mode="inner")
    code_img = np.where(bnd, lut[mip_lab], 0)
    for r, i in reason_code.items():
        m = code_img == i
        if m.any():
            rgba = np.zeros(m.shape + (4,)); rgba[m] = mcolors.to_rgba(QC_COLORS[r])
            ax.imshow(rgba, interpolation="nearest")
    counts = sub["qc_primary"].value_counts()
    for r, c in QC_COLORS.items():
        s = sub[sub["qc_primary"] == r]
        ax.scatter(s["cx_px"], s["cy_px"], s=6 if r == "pass" else 14, c=c, linewidths=0)
    ax.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=f"{r} (n={counts.get(r, 0)})")
                       for r, c in QC_COLORS.items()], loc="upper right", fontsize=8, framealpha=0.3)
    _deep = sub[~sub["qc_bad_not_deep"]]
    ax.set_title(f"QC at t={t} ({float(frame_to_hours(t)):.2f} {TIME_LABEL}); "
                 f"{_deep['qc_good_frame'].mean():.0%} of deep nuclei pass"); ax.set_xticks([]); ax.set_yticks([])
    crop_view(ax, crop, t)
    for row, (col, lo, hi, logx) in enumerate([
        (f"{BRIGHT_QC_CH}_mean_fluo", "qc_bright_lo", "qc_bright_hi", True),
        ("volume_um3", "qc_vol_lo", "qc_vol_hi", True),
        ("elongation", None, None, False)]):
        a = fig.add_subplot(gs[row, 1])
        v = sub.loc[~sub["qc_bad_not_deep"], col].dropna()
        v = v[v > 0]
        a.hist(np.log10(v) if logx else v, bins=60, color="#bbbbbb")
        cuts = [sub[lo].iloc[0], sub[hi].iloc[0]] if lo else [ELONG_MAX]
        for c in cuts:
            a.axvline(np.log10(c) if logx else c, color="#e06666", lw=1.5)
        a.set_xlabel(("log10 " if logx else "") + col); a.set_ylabel("deep nuclei")
    fig.tight_layout()
    return fig

fig = plot_qc_panel(QC_EXAMPLE_T)
save_fig(fig, f"04_qc_panel_t{QC_EXAMPLE_T}")
plt.show()

### Optional: cross-check pipeline features against an older parquet of the same movie

If you have an older table where `volume_um3` and `elongation` were computed from the masks directly and set `REFERENCE_PARQUET`, this cell checks that `volume_um3` agrees (should be ~identical) and how the pipeline's elongation maps onto the reference. It prints the elongation cutoff that excludes the same fraction as the reference's 3.5; it does not change `ELONG_MAX` for you.

In [ ]:
if REFERENCE_PARQUET is not None and Path(REFERENCE_PARQUET).exists():
    try:
        import pyarrow.parquet as pq
        _ref_cols = [c for c in ("t", "track_id", "volume_um3", "elongation") if c in pq.ParquetFile(REFERENCE_PARQUET).schema_arrow.names]
        ref = pd.read_parquet(REFERENCE_PARQUET, columns=_ref_cols)
        cmp_ = df_all[["t", "track_id", "volume_um3", "elongation"]].merge(ref, on=["t", "track_id"], suffixes=("", "_ref"))
        print(f"matched {len(cmp_):,} nucleus-frames with the reference parquet")
        out = {}
        for c in ("volume_um3", "elongation"):
            if f"{c}_ref" in cmp_:
                ok = cmp_[[c, f"{c}_ref"]].dropna()
                out[c] = {"spearman": ok[c].corr(ok[f"{c}_ref"], method="spearman"),
                          "median ratio new/ref": float((ok[c] / ok[f"{c}_ref"]).median())}
        display(pd.DataFrame(out))
        if "elongation_ref" in cmp_:
            frac_ref = (cmp_["elongation_ref"] > 3.5).mean()
            eq_cut = cmp_["elongation"].quantile(1 - frac_ref)
            print(f"reference elongation > 3.5 excludes {frac_ref:.2%}; the same fraction of the pipeline "
                  f"elongation is excluded by ELONG_MAX ≈ {eq_cut:.2f} (current ELONG_MAX = {ELONG_MAX})")
            samp = cmp_.dropna().sample(min(20000, len(cmp_.dropna())), random_state=0)
            fig = px.scatter(samp, x="elongation_ref", y="elongation", opacity=0.2,
                             title="Elongation: pipeline (1/elongation_ratio) vs reference")
            apply_dark_theme(fig, 14).show()
            log_step("4_reference_check", **out, equivalent_elong_cutoff=eq_cut, frac_ref=frac_ref)
    except Exception as exc:
        print("reference cross-check skipped:", exc)
else:
    print("no reference parquet; skipped")

## 5. Drift / registration check

**Question:** does the whole embryo move between frames? Whole-sample drift adds fake speed and persistence, and moves a "static" aggregate center.

The per-frame median displacement of all tracked deep nuclei is a drift estimate. Look for: steps or jumps (stage moves, re-registration), or a steady slope. Caveat: in a moving tissue the median also contains real collective flow, so decide `DRIFT_CORRECT` from the shape of the curve (abrupt = technical, smooth = maybe biology), not by default.

In [ ]:
def per_frame_drift(df):
    d = df.loc[df["frame_class"] == DEEP_CLASS_VALUE if DEEP_ONLY else slice(None),
               ["track_id", "t", "z_um", "y_um", "x_um"]].sort_values(["track_id", "t"])
    same = (d["track_id"].values[1:] == d["track_id"].values[:-1]) & (d["t"].values[1:] == d["t"].values[:-1] + 1)
    steps = pd.DataFrame(np.diff(d[["z_um", "y_um", "x_um"]].to_numpy(), axis=0)[same],
                         columns=["dz", "dy", "dx"])
    steps["t"] = d["t"].values[1:][same]
    med = steps.groupby("t")[["dz", "dy", "dx"]].median()
    full = pd.RangeIndex(int(df["t"].min()), int(df["t"].max()) + 1, name="t")
    med = med.reindex(full).fillna(0.0)
    cum = med.cumsum().rename(columns=lambda c: "cum_" + c)
    return pd.concat([med, cum], axis=1)

drift = per_frame_drift(df_all)
_step = np.sqrt((drift[["dz", "dy", "dx"]] ** 2).sum(axis=1))
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("cumulative median displacement (µm)", "per-frame median step (µm)"))
for c, col in zip(("cum_dz", "cum_dy", "cum_dx"), ("#6fa8dc", "#f6b26b", "#93c47d")):
    fig.add_trace(go.Scatter(x=frame_to_hours(drift.index), y=drift[c], name=c, line=dict(color=col)), 1, 1)
fig.add_trace(go.Scatter(x=frame_to_hours(drift.index), y=_step, name="|step|", line=dict(color="white")), 2, 1)
fig.update_xaxes(title_text=TIME_LABEL, row=2, col=1)
apply_dark_theme(fig, 14).update_layout(height=550, title="Drift check").show()
_jumps = drift.index[_step > _step.median() + 8 * (_step.quantile(0.75) - _step.quantile(0.25) + 1e-9)]
print(f"total median drift (z, y, x) µm: {drift[['cum_dz', 'cum_dy', 'cum_dx']].iloc[-1].round(1).tolist()}")
print(f"frames with outlier steps (possible stage jumps): {list(_jumps[:20])}{' ...' if len(_jumps) > 20 else ''}")

# positions used for motion parameters (drift-corrected only if chosen)
if DRIFT_CORRECT:
    _cum = drift[["cum_dz", "cum_dy", "cum_dx"]].reindex(df_all["t"]).to_numpy()
    df_all["zc_um"] = df_all["z_um"] - _cum[:, 0]
    df_all["yc_um"] = df_all["y_um"] - _cum[:, 1]
    df_all["xc_um"] = df_all["x_um"] - _cum[:, 2]
else:
    df_all["zc_um"], df_all["yc_um"], df_all["xc_um"] = df_all["z_um"], df_all["y_um"], df_all["x_um"]
log_step("5_drift", drift_correct=DRIFT_CORRECT, total_drift_um=drift[["cum_dz", "cum_dy", "cum_dx"]].iloc[-1].tolist(),
         jump_frames=list(_jumps))

## 6. Track gaps, divisions and eligibility

Motion parameters are computed **within one `track_id`**. At a division the parent track ends and two children start (`parent_track_id`), so a window that would cross a division is left empty (NaN) rather than stitched. Gaps (missing frames inside a track) also break windows.

`track_ok` = at least `MIN_TRACK_LEN_FRAMES` observations. Short tracks are kept but flagged; with a median track length near 18 frames, a 20-frame persistence window is only defined for a minority of tracks.

In [ ]:
T_FIRST, T_LAST = int(df_all["t"].min()), int(df_all["t"].max())
trk = df_all.groupby("track_id").agg(t_start=("t", "min"), t_end=("t", "max"), n_obs=("t", "size"),
                                     parent_track_id=("parent_track_id", "first"),
                                     not_deep_frac=("qc_bad_not_deep", "mean"),
                                     qc_pass_frac=("qc_good_frame", "mean"))
trk["n_gaps"] = (trk["t_end"] - trk["t_start"] + 1) - trk["n_obs"]
_children = df_all.loc[df_all["parent_track_id"] > 0].groupby("parent_track_id")["track_id"].nunique()
trk["n_children"] = _children.reindex(trk.index).fillna(0).astype(int)
trk["starts_from_division"] = trk["parent_track_id"] > 0
trk["ends_in_division"] = trk["n_children"] >= 2
trk["starts_at_movie_start"] = trk["t_start"] == T_FIRST
trk["ends_at_movie_end"] = trk["t_end"] == T_LAST
trk["track_ok"] = trk["n_obs"] >= MIN_TRACK_LEN_FRAMES
trk["end_reason"] = np.where(trk["ends_in_division"], "division", np.where(trk["ends_at_movie_end"], "movie_end", "lost"))
trk["censor_t"] = trk["t_end"]
df_all = df_all.drop(columns=[c for c in ("track_ok", "track_len") if c in df_all])
df_all = df_all.drop(columns=[c for c in ("end_reason",) if c in df_all])
df_all = df_all.join(trk[["track_ok", "n_obs", "end_reason"]].rename(columns={"n_obs": "track_len"}), on="track_id")

summary_tracks = {
    "tracks": len(trk), "median length (frames)": float(trk["n_obs"].median()),
    "tracks with gaps": f"{(trk['n_gaps'] > 0).mean():.1%}",
    "tracks ending in division": int(trk["ends_in_division"].sum()),
    "tracks starting from division": int(trk["starts_from_division"].sum()),
    f"track_ok (≥{MIN_TRACK_LEN_FRAMES} frames)": f"{trk['track_ok'].mean():.1%} of tracks, "
                                                  f"{df_all['track_ok'].mean():.1%} of nucleus-frames",
    **{f"nucleus-frames with a {w}-frame window possible": f"{(df_all['track_len'] > w).mean():.1%}"
       for w in PERSIST_WINDOWS},
}
display(pd.Series(summary_tracks, name="value").to_frame())
log_step("6_tracks", **summary_tracks)

## 7. Aggregate center (nuclear density)

The center is the **densest point of QC-passing nuclei**, found from nuclear positions only, so it is independent of any reporter (a reporter-defined center would make "reporter is higher near the aggregate" partly true by construction).

- **adaptive** (default): recomputed every frame, carried forward when a frame has fewer than `CENTER_MIN_N` nuclei, then smoothed with a rolling median over `CENTER_SMOOTH_FRAMES` so frame-to-frame jitter does not become fake radial motion.
- **static**: one center for the whole movie, from frame `STATIC_CENTER_T_REF`.

Two distance measures go downstream, both from the `CENTER_MODE` center:
- `r_um`: continuous distance (µm). Use this for models and correlations.
- `dist_bin` / `dist_bin_um`: `DIST_BIN_UM`-wide bins (`0–200`, `200–400`, …) for grouping and plots.

`r_um_adaptive` and `r_um_static` are both kept, so you can check that a result does not depend on the choice. Look at the trajectory plot: steps in the adaptive center mean the densest region changed, which may be biology (a new aggregate) or a segmentation/tracking change.

In [ ]:
def compute_center_density_from_df(df_t, radius_um=DENSITY_CENTER_RADIUS_UM, min_n=CENTER_MIN_N, use_qc=True):
    """Center (z, y, x) in µm of the densest neighbourhood of QC-passing nuclei in one frame."""
    sub = df_t[df_t["qc_good_frame"]] if use_qc else df_t
    if len(sub) < min_n:
        return None
    xyz = sub[["z_um", "y_um", "x_um"]].to_numpy(float)
    tree = cKDTree(xyz)
    counts = tree.query_ball_point(xyz, r=radius_um, return_length=True)
    peak = int(np.argmax(counts))
    return xyz[tree.query_ball_point(xyz[peak], r=radius_um)].mean(axis=0)

def compute_adaptive_centers_from_df(df, smooth_frames=CENTER_SMOOTH_FRAMES):
    rows, rules = {}, {}
    for t, df_t in df.groupby("t", sort=True):
        c = compute_center_density_from_df(df_t)
        rows[int(t)] = c if c is not None else [np.nan] * 3
        rules[int(t)] = "own" if c is not None else "carried"
    raw = pd.DataFrame.from_dict(rows, orient="index", columns=["cz_um", "cy_um", "cx_um"]).sort_index()
    raw.index.name = "t"
    raw["rule"] = pd.Series(rules)
    filled = raw[["cz_um", "cy_um", "cx_um"]].ffill().bfill()
    smooth = filled.rolling(smooth_frames, center=True, min_periods=1).median()
    return raw, smooth

def add_r_um(df, center_zyx_um, r_col="r_um_static"):
    c = np.asarray(center_zyx_um, float)
    df[r_col] = np.sqrt((df["z_um"] - c[0]) ** 2 + (df["y_um"] - c[1]) ** 2 + (df["x_um"] - c[2]) ** 2).astype(np.float32)
    return df

def add_r_um_adaptive(df, centers_by_t, r_col="r_um_adaptive"):
    c = centers_by_t.reindex(df["t"]).to_numpy()
    df[r_col] = np.sqrt((df["z_um"].to_numpy() - c[:, 0]) ** 2 + (df["y_um"].to_numpy() - c[:, 1]) ** 2
                        + (df["x_um"].to_numpy() - c[:, 2]) ** 2).astype(np.float32)
    return df

centers_raw, centers_adaptive = compute_adaptive_centers_from_df(df_all)
center_static_zyx_um = centers_adaptive.loc[min(STATIC_CENTER_T_REF, centers_adaptive.index.max())].to_numpy(float)
# centers actually used downstream (also for v_toward_agg in §8)
centers_mov = (centers_adaptive if CENTER_MODE == "adaptive"
               else pd.DataFrame([center_static_zyx_um] * len(centers_adaptive), index=centers_adaptive.index,
                                 columns=centers_adaptive.columns))
print(f"center mode: {CENTER_MODE} | static center (z, y, x µm) at t={STATIC_CENTER_T_REF}:",
      np.round(center_static_zyx_um, 1), f"| frames carried forward: {(centers_raw['rule'] == 'carried').sum()}")

df_all = add_r_um(df_all, center_static_zyx_um)
df_all = add_r_um_adaptive(df_all, centers_adaptive)
df_all["r_um"] = df_all["r_um_adaptive"] if CENTER_MODE == "adaptive" else df_all["r_um_static"]
df_all["dist_bin_um"] = ((df_all["r_um"] // DIST_BIN_UM) * DIST_BIN_UM).astype(np.float32)
df_all["dist_bin"] = [f"{int(b)}–{int(b + DIST_BIN_UM)}" if np.isfinite(b) else "nan" for b in df_all["dist_bin_um"]]
R_UM_DEFINITION = f"density center, {CENTER_MODE}"
display(df_all.groupby("dist_bin_um")["dist_bin"].agg(["first", "size"]).rename(columns={"first": "bin", "size": "n nucleus-frames"}))

# plots: center trajectory and adaptive-static distance
_h = frame_to_hours(centers_adaptive.index)
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("adaptive center (dots: per frame, line: smoothed), µm", "distance adaptive ↔ static center (µm)"))
for c, col in zip(("cz_um", "cy_um", "cx_um"), ("#6fa8dc", "#f6b26b", "#93c47d")):
    fig.add_trace(go.Scattergl(x=_h, y=centers_raw[c], mode="markers", marker=dict(size=2, color=col), showlegend=False), 1, 1)
    fig.add_trace(go.Scatter(x=_h, y=centers_adaptive[c], name=c, line=dict(color=col)), 1, 1)
fig.add_trace(go.Scatter(x=_h, y=np.linalg.norm(centers_adaptive.to_numpy() - center_static_zyx_um, axis=1),
                         name="|adaptive − static|", line=dict(color="white")), 2, 1)
fig.update_xaxes(title_text=TIME_LABEL, row=2, col=1)
apply_dark_theme(fig, 14).update_layout(height=600, title="Aggregate center (nuclear density)").show()

# overlay on the view frame
fig, ax = plt.subplots(figsize=(7, 7))
_img = read_mip_frame(NUCLEAR_CH, VIEW_T)
ax.imshow(_img, cmap="gray", clim=contrast_limits(_img), extent=mip_on_mask_grid_extent())
_trail = centers_adaptive.loc[:VIEW_T]
ax.plot(_trail["cx_um"] / scale_vec[2], _trail["cy_um"] / scale_vec[1], color="#f6b26b", lw=1, label="adaptive center path")
ax.scatter([_trail["cx_um"].iloc[-1] / scale_vec[2]], [_trail["cy_um"].iloc[-1] / scale_vec[1]], s=80, c="#f6b26b", label=f"adaptive, t={VIEW_T}")
ax.scatter([center_static_zyx_um[2] / scale_vec[2]], [center_static_zyx_um[1] / scale_vec[1]], s=80, marker="x", c="#e06666", label="static")
_cy, _cx = _trail["cy_um"].iloc[-1] / scale_vec[1], _trail["cx_um"].iloc[-1] / scale_vec[2]
for k in range(1, 4):   # distance-bin rings (x-y projection of the 3D distance)
    ax.add_patch(plt.Circle((_cx, _cy), k * DIST_BIN_UM / scale_vec[2], fill=False, ls=":", color="#f6b26b", lw=0.8))
ax.legend(fontsize=8); ax.set_xticks([]); ax.set_yticks([]); ax.set_title(f"Aggregate center on nuclear MIP, t={VIEW_T}; rings every {DIST_BIN_UM} µm")
crop_view(ax, VIEW_CROP_YX, VIEW_T)
save_fig(fig, f"07_centers_t{VIEW_T}")
plt.show()
log_step("7_center", center_mode=CENTER_MODE, static_center_um=center_static_zyx_um, static_t_ref=STATIC_CENTER_T_REF,
         r_um_definition=R_UM_DEFINITION, frames_carried=int((centers_raw["rule"] == "carried").sum()),
         density_radius_um=DENSITY_CENTER_RADIUS_UM, dist_bin_um=DIST_BIN_UM)
centers_export = pd.concat([centers_raw.add_suffix("_raw"), centers_adaptive], axis=1).reset_index()
centers_export.to_csv(OUT_DIR / "aggregate_centers_per_frame.csv", index=False)

## 8. Per-nucleus parameters

All in µm and minutes, computed within a track (see §6), centred on frame `t`:

| column | definition |
|---|---|
| `r_um` (+ `r_um_adaptive`, `r_um_static`), `dist_bin` | distance to the aggregate center, continuous and binned (§7) |
| `speed_um_min` | path length over t ± `SPEED_HALF_WINDOW` / elapsed time (includes localisation jitter: slow cells read slightly fast) |
| `speed_xy_um_min` | same, x-y only (z voxels are 3 µm, so z jitter can dominate 3D speed); for smoother tracks set `TRACKS_PATH` to `tracks_smoothed.csv` |
| `persist_w{W}` | straightness = net displacement / path length over a W-frame window (0 = random, 1 = straight). Jitter lowers it for slow cells. `persistence` = `persist_w{PERSIST_WINDOW_MAIN}` |
| `v_toward_agg_um_min` | rate of approach to the aggregate center over the main window (+ = moving toward), measured against the center at `t` so center motion is not counted |
| `density_n` | other nuclei within `DENSITY_RADIUS_UM` (`DENSITY_POPULATION`) |
| `nn_dist_um` | distance to the nearest other nucleus |
| `depth_um` | depth below a local surface proxy: 5th percentile of nuclear z within the same `DEPTH_TILE_UM` x-y tile and frame (`DEPTH_SURFACE` says which side faces the objective) |
| `*_mean_fluo` | pipeline mean intensity per channel (raw, own scale) |

In [ ]:
def _offset_lookup(df, cols, k):
    """Values of `cols` for the same track at frame t + k (NaN if absent)."""
    src = df.set_index(["track_id", "t"])[cols]
    idx = pd.MultiIndex.from_arrays([df["track_id"].to_numpy(), df["t"].to_numpy() + k])
    return src.reindex(idx).to_numpy()

def add_motion_params(df):
    P = ["zc_um", "yc_um", "xc_um"]
    p0 = df[P].to_numpy(float)
    prev = _offset_lookup(df, P, -1)
    step = np.linalg.norm(p0 - prev, axis=1)              # NaN at track start or after a gap
    df["step_um"] = step.astype(np.float32)
    tmp = df[["track_id", "t"]].copy()
    tmp["s0"] = np.nan_to_num(step)
    tmp["gap"] = np.isnan(step).astype(int)
    tmp = tmp.sort_values(["track_id", "t"])
    tmp["cs"] = tmp.groupby("track_id")["s0"].cumsum()
    tmp["cg"] = tmp.groupby("track_id")["gap"].cumsum()
    tmp = tmp.sort_index()
    df["_cs"], df["_cg"] = tmp["cs"], tmp["cg"]

    def window(h):
        lo = _offset_lookup(df, P + ["_cs", "_cg"], -h)
        hi = _offset_lookup(df, P + ["_cs", "_cg"], +h)
        net = np.linalg.norm(hi[:, :3] - lo[:, :3], axis=1)
        path = hi[:, 3] - lo[:, 3]
        ok = (hi[:, 4] - lo[:, 4]) == 0                   # no gap inside the window
        net[~ok] = np.nan; path[~ok] = np.nan
        return net, path

    _, path_s = window(SPEED_HALF_WINDOW)
    df["speed_um_min"] = (path_s / (2 * SPEED_HALF_WINDOW * DT_MIN)).astype(np.float32)
    # xy-only speed: z steps are 3 µm voxels, so z jitter can dominate 3D speed
    stepxy = np.linalg.norm(df[["yc_um", "xc_um"]].to_numpy(float) - _offset_lookup(df, ["yc_um", "xc_um"], -1), axis=1)
    tmp["sxy"] = pd.Series(np.nan_to_num(stepxy), index=df.index).reindex(tmp.index)
    df["_csxy"] = tmp.sort_values(["track_id", "t"]).groupby("track_id")["sxy"].cumsum().sort_index()
    h = SPEED_HALF_WINDOW
    lo = _offset_lookup(df, ["_csxy", "_cg"], -h); hi = _offset_lookup(df, ["_csxy", "_cg"], +h)
    pxy = np.where((hi[:, 1] - lo[:, 1]) == 0, hi[:, 0] - lo[:, 0], np.nan)
    df["speed_xy_um_min"] = (pxy / (2 * h * DT_MIN)).astype(np.float32)
    for W in PERSIST_WINDOWS:
        net, path = window(W // 2)
        with np.errstate(invalid="ignore", divide="ignore"):
            df[f"persist_w{W}"] = np.where(path > 0, net / path, np.nan).astype(np.float32)
    df["persistence"] = df[f"persist_w{PERSIST_WINDOW_MAIN}"]

    # approach rate to the moving center (raw positions, center fixed at frame t)
    h = PERSIST_WINDOW_MAIN // 2
    c = centers_mov.reindex(df["t"]).to_numpy()
    R = ["z_um", "y_um", "x_um"]
    lo = _offset_lookup(df, R, -h); hi = _offset_lookup(df, R, +h)
    df["v_toward_agg_um_min"] = (-(np.linalg.norm(hi - c, axis=1) - np.linalg.norm(lo - c, axis=1))
                                 / (2 * h * DT_MIN)).astype(np.float32)
    return df.drop(columns=["_cs", "_cg", "_csxy"])

def add_neighbourhood_params(df):
    dens = np.full(len(df), np.nan, np.float32); nn = np.full(len(df), np.nan, np.float32)
    pos = df.index.get_indexer
    for t, g in df.groupby("t"):
        pop = g if DENSITY_POPULATION == "all" else g[~g["qc_bad_not_deep"]]
        if len(pop) < 2:
            continue
        tree = cKDTree(pop[["z_um", "y_um", "x_um"]].to_numpy(float))
        q = g[["z_um", "y_um", "x_um"]].to_numpy(float)
        in_pop = g.index.isin(pop.index)
        ii = pos(g.index)
        dens[ii] = tree.query_ball_point(q, r=DENSITY_RADIUS_UM, return_length=True) - in_pop
        d, _ = tree.query(q, k=2)
        nn[ii] = np.where(in_pop, d[:, 1], d[:, 0])
    df["density_n"], df["nn_dist_um"] = dens, nn
    # local surface: shallowest nuclei within each DEPTH_TILE_UM x-y tile (the blastoderm is curved)
    tile = ((df["y_um"] // DEPTH_TILE_UM).astype(np.int64) * 100000 + (df["x_um"] // DEPTH_TILE_UM).astype(np.int64))
    if DEPTH_SURFACE == "min_z":
        surf = df.groupby([df["t"], tile])["z_um"].transform(lambda z: z.quantile(0.05))
        df["depth_um"] = (df["z_um"] - surf).astype(np.float32)
    else:
        surf = df.groupby([df["t"], tile])["z_um"].transform(lambda z: z.quantile(0.95))
        df["depth_um"] = (surf - df["z_um"]).astype(np.float32)
    return df

df_all = add_motion_params(df_all)
df_all = add_neighbourhood_params(df_all)
PARAM_COLS = ["r_um", "speed_um_min", "speed_xy_um_min", "persistence", "v_toward_agg_um_min", "density_n", "nn_dist_um",
              "depth_um", "volume_um3"] + FLUO_COLS
display(df_all.loc[df_all["qc_good_frame"] & df_all["track_ok"], PARAM_COLS].describe(percentiles=[0.05, 0.5, 0.95]).T.round(3))
log_step("8_params", speed_half_window=SPEED_HALF_WINDOW, persist_windows=PERSIST_WINDOWS,
         persist_main=PERSIST_WINDOW_MAIN, density_radius_um=DENSITY_RADIUS_UM,
         density_population=DENSITY_POPULATION, depth_surface=DEPTH_SURFACE,
         defined_frac={c: float(df_all[c].notna().mean()) for c in PARAM_COLS})

## 9. Averages over time

Median (line) and interquartile range (band) per `TIME_BIN_FRAMES`-frame bin, for QC-passing nuclei on `track_ok` tracks. The bottom panel is the count per bin: sparse bins make medians jump. Set `AVG_GROUP_BY = "dist_bin"` to split the lines by distance bin; to split by reporter status, set `AVG_GROUP_BY = f"{SECOND_CH}_status"` and rerun this cell after §11.

In [ ]:
def time_summary(df, cols, group=None, bin_frames=TIME_BIN_FRAMES):
    """Per time bin (and group): q25 / median / q75 of each column, plus n. Flat columns."""
    d = df[df["qc_good_frame"] & df["track_ok"]].copy()
    d["t_bin"] = (d["t"] // bin_frames) * bin_frames + bin_frames / 2
    keys = ["t_bin"] + ([group] if group else [])
    g = d.groupby(keys, observed=True)
    parts = [g.size().rename("n")]
    for c in cols:
        parts += [g[c].quantile(0.25).rename(f"{c}_q25"), g[c].median().rename(f"{c}_med"),
                  g[c].quantile(0.75).rename(f"{c}_q75")]
    return pd.concat(parts, axis=1).reset_index()

def plot_time_summary(summ, cols, group=None, title="Parameters over time"):
    fig = make_subplots(rows=len(cols) + 1, cols=1, shared_xaxes=True, vertical_spacing=0.015,
                        subplot_titles=cols + ["n per bin"])
    groups = [("all", summ)] if group is None else list(summ.groupby(group))
    for gi, (gname, s) in enumerate(groups):
        col = SEGMENT_COLORS[gi % len(SEGMENT_COLORS)] if group else "#d9d9d9"
        x = frame_to_hours(s["t_bin"].to_numpy())
        for i, c in enumerate(cols, start=1):
            fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[s[f"{c}_q75"].to_numpy(), s[f"{c}_q25"].to_numpy()[::-1]],
                                     fill="toself", fillcolor=col, opacity=0.2, line=dict(width=0),
                                     showlegend=False, hoverinfo="skip"), i, 1)
            fig.add_trace(go.Scatter(x=x, y=s[f"{c}_med"], line=dict(color=col), name=str(gname),
                                     showlegend=(i == 1)), i, 1)
        fig.add_trace(go.Scatter(x=x, y=s["n"], line=dict(color=col), showlegend=False), len(cols) + 1, 1)
    fig.update_xaxes(title_text=TIME_LABEL, row=len(cols) + 1, col=1)
    return apply_dark_theme(fig, 12).update_layout(height=190 * (len(cols) + 1), title=title)

AVG_COLS = ["r_um", "speed_um_min", "persistence", "v_toward_agg_um_min", "density_n", NUC_COL] + ([CH2_COL] if CH2_COL else [])
_grp = AVG_GROUP_BY if (AVG_GROUP_BY in df_all.columns) else None
summ = time_summary(df_all, AVG_COLS, group=_grp)
fig = plot_time_summary(summ, AVG_COLS, group=_grp)
save_fig(fig, "09_params_over_time"); fig.show()

# persistence-window sensitivity (#4)
_pcols = [f"persist_w{w}" for w in PERSIST_WINDOWS]
summ_p = time_summary(df_all, _pcols)
fig = go.Figure()
for i, c in enumerate(_pcols):
    fig.add_trace(go.Scatter(x=frame_to_hours(summ_p["t_bin"]), y=summ_p[f"{c}_med"], name=f"{c} ({int(c.split('w')[1]) * DT_MIN:.0f} min)",
                             line=dict(color=SEGMENT_COLORS[i])))
apply_dark_theme(fig, 14).update_layout(title="Persistence: sensitivity to window length (median)",
                                        xaxis_title=TIME_LABEL, yaxis_title="net / path").show()
print("Longer windows read straighter for the same motion and exist for fewer tracks; "
      "compare shapes over time, not absolute levels across windows.")

## 10. Depth check

**Question:** does intensity fall with depth? In lightsheet data, scattering dims deep nuclei, and the aggregate may sit at a different depth from the periphery, so a depth gradient can look like a spatial pattern.

Look for: a falling nuclear-channel (control) curve. If the control falls as much as the second channel, depth should be a covariate in any spatial comparison of the second channel.

In [ ]:
_d = df_all[df_all["qc_good_frame"]].copy()
_d["depth_bin"] = (_d["depth_um"] // 20) * 20 + 10
fig = make_subplots(rows=1, cols=len(FLUO_COLS), subplot_titles=FLUO_COLS)
_depth_drop = {}
for j, fc in enumerate(FLUO_COLS, start=1):
    for i, (seg, g) in enumerate(_d.groupby("movie_segment")):
        prof = g.groupby("depth_bin")[fc].agg(["median", "size"])
        prof = prof[prof["size"] >= 30]
        fig.add_trace(go.Scatter(x=prof.index, y=prof["median"], name=seg, line=dict(color=SEGMENT_COLORS[i]),
                                 showlegend=(j == 1)), 1, j)
    prof = _d.groupby("depth_bin")[fc].agg(["median", "size"]).query("size >= 30")["median"]
    if len(prof) > 2:
        _depth_drop[fc] = float(1 - prof.iloc[-1] / prof.iloc[0])
fig.update_xaxes(title_text="depth below surface proxy (µm)")
apply_dark_theme(fig, 14).update_layout(title="Median intensity vs depth (QC-passing; per movie segment)").show()
print("fractional drop from shallowest to deepest bin:", {k: round(v, 2) for k, v in _depth_drop.items()})
log_step("10_depth", depth_drop=_depth_drop)

## 11. Second channel: distributions and threshold rules

Runs only if `SECOND_CH` is set and present.

**11a. Distributions.** One histogram per movie segment (thirds by default), normalised to density, QC-passing deep nuclei. A shift between segments means a single fixed threshold will mark a different fraction of nuclei over time, which may be the biology (the reporter turning on) or a global brightness change (bleaching, depth, laser). The nuclear-channel histograms are shown for comparison.

**11b. Threshold rules** (`THRESHOLD_MODE`):
- `fixed` (default): one value for the whole movie. **There is no default value**: set `FIXED_THRESHOLD` after looking at the histograms above (they print suggested starting points). Until it is set, the status columns, gallery and status colours are skipped and the rest of the notebook still runs.
- `adaptive`: one value per frame from that frame's distribution (`ADAPTIVE_METHOD`: quantile, Otsu, or median + k·MAD), smoothed over `ADAPTIVE_SMOOTH_FRAMES`. Follows global drifts in brightness.
- `percentile`: top `PERCENTILE_TOP` % in each frame. **This fixes the positive fraction by construction**: it cannot show the positive population growing, only which nuclei are in the top tail.

Every mode then requires `MIN_CONSECUTIVE_FRAMES` consecutive above-threshold frames within a track ("top 10 % for 3 frames in a row"). Caveat: a run that is cut off by the end of a track is not counted even if it might have continued.

**Control:** the same rule is applied to the nuclear channel. For `fixed`, the control threshold is the nuclear-channel quantile that matches the overall positive fraction. If the control's positive fraction tracks the second channel's over time, or the same nuclei are positive in both, "positive" may reflect nuclear size, import or depth rather than second-channel biology.

In [ ]:
STATUS_READY = bool(CH2_COL) and not (THRESHOLD_MODE == "fixed" and FIXED_THRESHOLD is None)
if CH2_COL:
    pop2 = df_all[df_all["qc_good_frame"]]
    fig = make_subplots(rows=1, cols=2, subplot_titles=(f"{SECOND_CH}", f"{NUCLEAR_CH} (control)"))
    for j, col in enumerate([CH2_COL, NUC_COL], start=1):
        for i, (seg, g) in enumerate(pop2.groupby("movie_segment")):
            v = g[col].dropna(); v = v[v > 0]
            hrs = frame_to_hours([g["t"].min(), g["t"].max()])
            fig.add_trace(go.Histogram(x=np.log10(v) if HIST_LOG10 else v, histnorm="probability density", nbinsx=120,
                                       name=f"{seg}: {hrs[0]:.1f}–{hrs[1]:.1f} h", opacity=0.45,
                                       marker_color=SEGMENT_COLORS[i], showlegend=(j == 1)), 1, j)
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title_text=("log10 " if HIST_LOG10 else "") + "mean intensity")
    if THRESHOLD_MODE == "fixed" and FIXED_THRESHOLD is not None:
        fig.add_vline(x=np.log10(FIXED_THRESHOLD) if HIST_LOG10 else FIXED_THRESHOLD, line_color="#e06666", row=1, col=1)
    fig = apply_dark_theme(fig, 14).update_layout(title="Intensity distributions per movie segment (QC-passing)")
    save_fig(fig, "11_histograms_by_segment"); fig.show()
    _v = pop2[CH2_COL].dropna()
    _sug = {"otsu (whole movie)": float(threshold_otsu(_v.to_numpy())),
            "90th percentile, first segment": float(pop2.loc[pop2["movie_segment"] == "seg1", CH2_COL].quantile(0.9))}
    print(f"{SECOND_CH} mean_fluo: median {_v.median():.1f}, 5–95 % {_v.quantile(0.05):.1f}–{_v.quantile(0.95):.1f}")
    print("starting points for FIXED_THRESHOLD (not defaults; check against the histograms and the §12 gallery):",
          {k: round(v, 1) for k, v in _sug.items()})
    if not STATUS_READY:
        print(f"FIXED_THRESHOLD is not set: {SECOND_CH} status will be skipped until you set it in CONFIG and rerun from §11.")
else:
    print("no second channel: section 11 skipped")

In [ ]:
def per_frame_threshold(df, value_col, mode=THRESHOLD_MODE, fixed=FIXED_THRESHOLD):
    """Threshold for every frame (pd.Series indexed by t), computed on QC-passing nuclei."""
    pop = df[df["qc_good_frame"]].dropna(subset=[value_col])
    frames = pd.RangeIndex(int(df["t"].min()), int(df["t"].max()) + 1, name="t")
    if mode == "fixed":
        return pd.Series(float(fixed), index=frames)
    g = pop.groupby("t")[value_col]
    if mode == "percentile":
        thr = g.quantile(1 - PERCENTILE_TOP / 100)
        return thr.reindex(frames).interpolate(limit_direction="both")
    if mode == "adaptive":
        if ADAPTIVE_METHOD == "quantile":
            raw = g.quantile(ADAPTIVE_QUANTILE)
        elif ADAPTIVE_METHOD == "otsu":
            raw = g.apply(lambda v: threshold_otsu(v.to_numpy()) if v.nunique() > 2 else np.nan)
        elif ADAPTIVE_METHOD == "median_mad":
            raw = g.median() + ADAPTIVE_MAD_K * 1.4826 * g.apply(lambda v: (v - v.median()).abs().median())
        else:
            raise ValueError(ADAPTIVE_METHOD)
        raw = raw.reindex(frames).interpolate(limit_direction="both")
        return raw.rolling(ADAPTIVE_SMOOTH_FRAMES, center=True, min_periods=1).median()
    raise ValueError(mode)

def apply_persistence(df, above, k=MIN_CONSECUTIVE_FRAMES):
    """True for frames inside a within-track run of >= k consecutive above-threshold frames.
    Also returns the run start frame for each positive row."""
    order = np.lexsort((df["t"].to_numpy(), df["track_id"].to_numpy()))
    a = np.asarray(above, bool)[order]
    tr = df["track_id"].to_numpy()[order]; t = df["t"].to_numpy()[order]
    new = np.ones(len(a), bool)
    new[1:] = (tr[1:] != tr[:-1]) | (t[1:] != t[:-1] + 1) | (a[1:] != a[:-1])
    run = np.cumsum(new)
    run_len = np.bincount(run)[run]
    run_start = t[np.flatnonzero(new)][run - 1]
    pos = a & (run_len >= k)
    out_pos = np.empty(len(a), bool); out_pos[order] = pos
    out_start = np.full(len(a), -1, np.int64); out_start[order] = np.where(pos, run_start, -1)
    return out_pos, out_start

def classify_channel(df, value_col, thr_series, k=MIN_CONSECUTIVE_FRAMES):
    thr = thr_series.reindex(df["t"]).to_numpy()
    v = df[value_col].to_numpy()
    above = np.nan_to_num(v, nan=-np.inf) >= thr
    pos, start = apply_persistence(df, above, k)
    return thr, above, pos, start

if STATUS_READY:
    thr_ch2 = per_frame_threshold(df_all, CH2_COL)
    thr, above, pos, start = classify_channel(df_all, CH2_COL, thr_ch2)
    df_all[f"{SECOND_CH}_thr"] = thr.astype(np.float32)
    df_all[f"{SECOND_CH}_above_thr"] = above
    df_all[f"{SECOND_CH}_positive"] = pos
    df_all[f"{SECOND_CH}_status"] = np.where(df_all[CH2_COL].isna(), "none", np.where(pos, "positive", "negative"))

    # control: same rule on the nuclear channel
    if THRESHOLD_MODE == "fixed":
        _pp = df_all[df_all["qc_good_frame"]]
        _q = (_pp[CH2_COL] < FIXED_THRESHOLD).mean()
        thr_nuc = per_frame_threshold(df_all, NUC_COL, mode="fixed", fixed=_pp[NUC_COL].quantile(_q))
    else:
        thr_nuc = per_frame_threshold(df_all, NUC_COL)
    _, _, pos_c, _ = classify_channel(df_all, NUC_COL, thr_nuc)
    df_all[f"{NUCLEAR_CH}_positive_control"] = pos_c

    # plots: threshold vs per-frame distribution; positive fraction over time (both channels)
    pop2 = df_all[df_all["qc_good_frame"]]
    qtl = pop2.groupby("t")[CH2_COL].quantile([0.05, 0.5, 0.95]).unstack()
    frac = pop2.groupby("t")[[f"{SECOND_CH}_positive", f"{NUCLEAR_CH}_positive_control"]].mean()
    hh = frame_to_hours(qtl.index)
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=(f"{SECOND_CH}: per-frame 5/50/95 % and threshold", "fraction positive (QC-passing)"))
    fig.add_trace(go.Scatter(x=np.r_[hh, hh[::-1]], y=np.r_[qtl[0.95], qtl[0.05][::-1]], fill="toself",
                             fillcolor="#6fa8dc", opacity=0.25, line=dict(width=0), name="5–95 %"), 1, 1)
    fig.add_trace(go.Scatter(x=hh, y=qtl[0.5], line=dict(color="#6fa8dc"), name="median"), 1, 1)
    fig.add_trace(go.Scatter(x=frame_to_hours(thr_ch2.index), y=thr_ch2, line=dict(color="#e06666"), name="threshold"), 1, 1)
    fig.add_trace(go.Scatter(x=frame_to_hours(frac.index), y=frac.iloc[:, 0], line=dict(color=STATUS_COLORS["positive"]), name=f"{SECOND_CH} positive"), 2, 1)
    fig.add_trace(go.Scatter(x=frame_to_hours(frac.index), y=frac.iloc[:, 1], line=dict(color="#bbbbbb", dash="dot"), name=f"{NUCLEAR_CH} control"), 2, 1)
    fig.update_xaxes(title_text=TIME_LABEL, row=2, col=1)
    fig = apply_dark_theme(fig, 13).update_layout(height=600, title=f"Threshold rule: {THRESHOLD_MODE}, ≥{MIN_CONSECUTIVE_FRAMES} consecutive frames")
    save_fig(fig, "11_threshold_rule"); fig.show()

    both = pop2[f"{SECOND_CH}_positive"] & pop2[f"{NUCLEAR_CH}_positive_control"]
    either = pop2[f"{SECOND_CH}_positive"] | pop2[f"{NUCLEAR_CH}_positive_control"]
    ctrl = {"corr_of_positive_fraction_over_time": float(frac.iloc[:, 0].corr(frac.iloc[:, 1])),
            "jaccard_positive_nuclei": float(both.sum() / max(either.sum(), 1)),
            "positive_frac_overall": float(pop2[f"{SECOND_CH}_positive"].mean())}
    if THRESHOLD_MODE == "percentile":
        ctrl["corr_of_positive_fraction_over_time"] = None   # both fractions are fixed by construction
        print("percentile mode: positive fractions are fixed by construction, so only the overlap "
              "(jaccard) between channels is informative here.")
    print(ctrl)
    if ctrl["jaccard_positive_nuclei"] > 0.5:
        print("WARNING: most second-channel-positive nuclei are also positive by the same rule in the nuclear "
              "channel. 'Positive' may be tracking nuclear brightness (size, import, depth).")
    log_step("11_threshold", mode=THRESHOLD_MODE, fixed=FIXED_THRESHOLD, adaptive_method=ADAPTIVE_METHOD,
             adaptive_q=ADAPTIVE_QUANTILE, percentile_top=PERCENTILE_TOP, k=MIN_CONSECUTIVE_FRAMES,
             threshold_range=(float(thr_ch2.min()), float(thr_ch2.max())), **ctrl)

### 11c. Track-level status and event-time columns

- `{ch}_status` (per frame): `positive` / `negative` / `none` (no intensity). This is the parameter "second-channel condition".
- `{ch}_track_ever_positive`: the track has at least one positive run.
- `{ch}_first_positive_t` / `_hours`: first frame of the first positive run (the event).
- `{ch}_r_um_at_first_positive`, `{ch}_dist_bin_at_first_positive`: where the nucleus was at the event, for event-time spatial analysis (rather than whole-track summaries).
- `t_rel_first_positive`: frames relative to the event, for event-triggered averages.
- `end_reason` (per track: `division`, `movie_end`, `lost`) and `censor_t` (last observed frame): never-positive tracks are right-censored at `censor_t`, not "never".

A track that starts already positive gets an event at its first frame; that is left-censored, not an onset. `{ch}_onset_observed` requires at least `ONSET_MIN_NEG_FRAMES` QC-passing negative frames before the event. Daughter tracks start at division with no history of their own, so their onsets are often left-censored; a lineage-level onset (following `parent_track_id`) is not computed here, because lineages would need manual track correction first.

In [ ]:
if STATUS_READY:
    S = SECOND_CH
    _p = df_all.loc[df_all[f"{S}_positive"], ["track_id", "t"]]
    first = _p.groupby("track_id")["t"].min().rename(f"{S}_first_positive_t")
    ev = (first.reset_index().rename(columns={f"{S}_first_positive_t": "t"})
          .merge(df_all[["track_id", "t", "r_um", "dist_bin"]], on=["track_id", "t"], how="left")
          .set_index("track_id")[["r_um", "dist_bin"]])
    tl = trk[["t_start"]].join(first).join(ev.rename(columns={"r_um": f"{S}_r_um_at_first_positive",
                                                               "dist_bin": f"{S}_dist_bin_at_first_positive"}))
    tl[f"{S}_track_ever_positive"] = tl[f"{S}_first_positive_t"].notna()
    tl[f"{S}_first_positive_hours"] = frame_to_hours(tl[f"{S}_first_positive_t"])
    # observed onset: >= ONSET_MIN_NEG_FRAMES QC-passing negative frames before the event
    _pre = df_all[["track_id", "t", "qc_good_frame", f"{S}_positive"]].join(first, on="track_id")
    _pre = _pre[(_pre["t"] < _pre[f"{S}_first_positive_t"]) & _pre["qc_good_frame"] & ~_pre[f"{S}_positive"]]
    tl[f"{S}_n_neg_frames_before"] = _pre.groupby("track_id").size().reindex(tl.index).fillna(0).astype(int)
    tl[f"{S}_onset_observed"] = tl[f"{S}_track_ever_positive"] & (tl[f"{S}_n_neg_frames_before"] >= ONSET_MIN_NEG_FRAMES)
    _newcols = [c for c in tl.columns if c != "t_start"]
    df_all = df_all.drop(columns=[c for c in _newcols if c in df_all]).join(tl[_newcols], on="track_id")
    df_all["t_rel_first_positive"] = (df_all["t"] - df_all[f"{S}_first_positive_t"]).astype("float32")
    trk = trk.drop(columns=[c for c in _newcols if c in trk]).join(tl[_newcols])
    print(f"tracks ever positive: {tl[f'{S}_track_ever_positive'].sum()} of {len(tl)} "
          f"({tl[f'{S}_onset_observed'].sum()} with an observed onset)")
    display(trk.loc[trk[f"{S}_onset_observed"], [f"{S}_first_positive_hours", f"{S}_r_um_at_first_positive"]].describe().round(2))

## 12. Positive-call QC gallery

Crops at `GALLERY_T` around randomly chosen positive, borderline (within ±`BORDERLINE_FRAC` of the threshold) and negative QC-passing nuclei. Second-channel and nuclear crops share contrast within each channel across all crops. Titles: intensity / threshold, depth, and `occl` if the label MIP at the nucleus centroid belongs to a different nucleus (it is covered in the projection, so the MIP crop shows its neighbour; judge with care).

Look for: positives that look clearly brighter in the second channel but not systematically larger or brighter in the nuclear channel; borderlines that are genuinely ambiguous.

In [ ]:
def gallery(t, n=GALLERY_N, seed=0):
    sub = df_all[(df_all["t"] == t) & df_all["qc_good_frame"]].copy()
    ratio = sub[CH2_COL] / sub[f"{SECOND_CH}_thr"]
    groups = {
        "positive": sub[sub[f"{SECOND_CH}_positive"]],
        "borderline": sub[(ratio - 1).abs() < BORDERLINE_FRAC],
        "negative": sub[(~sub[f"{SECOND_CH}_positive"]) & (ratio < 1 - BORDERLINE_FRAC)],
    }
    mip_lab = label_mip(t)
    imgs = {ch: read_mip_frame(ch, t) for ch in (SECOND_CH, NUCLEAR_CH) if ch in MIP_PATHS}
    clims = {ch: contrast_limits(im) for ch, im in imgs.items()}
    fig, axes = plt.subplots(len(groups) * len(imgs), n, figsize=(1.9 * n, 2.0 * len(groups) * len(imgs)), squeeze=False)
    for gi, (gname, g) in enumerate(groups.items()):
        pick = g.sample(min(n, len(g)), random_state=seed) if len(g) else g
        for ci, (ch, im) in enumerate(imgs.items()):
            row = gi * len(imgs) + ci
            sy, sx = im.shape[0] / Y_MASK, im.shape[1] / X_MASK
            hy = int(GALLERY_CROP_UM / 2 / scale_vec[1] * sy); hx = int(GALLERY_CROP_UM / 2 / scale_vec[2] * sx)
            for j in range(n):
                ax = axes[row, j]; ax.set_xticks([]); ax.set_yticks([])
                if j == 0:
                    ax.set_ylabel(f"{gname}\n{ch}", fontsize=8)
                if j >= len(pick):
                    continue
                r = pick.iloc[j]
                cy, cx = int(r["cy_px"] * sy), int(r["cx_px"] * sx)
                crop = im[max(cy - hy, 0):cy + hy, max(cx - hx, 0):cx + hx]
                ax.imshow(crop, cmap="gray", clim=clims[ch])
                ax.axhline(min(hy, cy), color="#e06666", lw=0.4); ax.axvline(min(hx, cx), color="#e06666", lw=0.4)
                if ci == 0:
                    occl = mip_lab[int(r["cy_px"]), int(r["cx_px"])] != r["track_id"]
                    ax.set_title(f"{r[CH2_COL] / r[f'{SECOND_CH}_thr']:.2f}× d{r['depth_um']:.0f}{' occl' if occl else ''}", fontsize=7)
    fig.suptitle(f"Positive-call gallery, t={t} ({THRESHOLD_MODE}, ≥{MIN_CONSECUTIVE_FRAMES} frames)", fontsize=10)
    fig.tight_layout()
    return fig

if STATUS_READY and SECOND_CH in MIP_PATHS:
    fig = gallery(GALLERY_T)
    save_fig(fig, f"12_positive_gallery_t{GALLERY_T}")
    plt.show()

## 13. Parquet builder

Writes, to `OUT_DIR`:
- `{dataset}_nuclei_{version}_{stamp}.parquet`: one row per nucleus per frame, **all rows kept**, with QC flags and `qc_reason` so exclusions stay auditable. Provenance (inputs with sizes and modification times, git commit, every CONFIG value, run log) is embedded in the parquet schema metadata under `provenance`.
- `{dataset}_tracks_{version}_{stamp}.parquet`: one row per track.
- `{dataset}_params_{version}_{stamp}.json`: the same provenance as a readable sidecar.
- `../embryo_summary.csv`: one row per embryo (replaced if this dataset was exported before). Across movies, the embryo is the replicate.

Read the provenance back with: `json.loads(pq.read_schema(path).metadata[b"provenance"])`.

In [ ]:
def git_commit():
    try:
        sha = subprocess.run(["git", "-C", str(PROJECT_ROOT), "rev-parse", "--short", "HEAD"],
                             capture_output=True, text=True, timeout=10).stdout.strip() or None
        dirty = bool(subprocess.run(["git", "-C", str(PROJECT_ROOT), "status", "--porcelain"],
                                    capture_output=True, text=True, timeout=10).stdout.strip())
        return {"commit": sha, "uncommitted_changes": dirty}
    except Exception:
        return None

def package_versions():
    import importlib
    out = {"python": sys.version.split()[0]}
    for m in ("numpy", "pandas", "scipy", "skimage", "zarr", "pyarrow", "plotly", "matplotlib"):
        try:
            out[m] = importlib.import_module(m).__version__
        except Exception:
            pass
    return out

stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
provenance = {
    "dataset": DATASET_NAME, "embryo_id": EMBRYO_ID, "created": stamp,
    "notebook": "movie_explorer.ipynb", "git": git_commit(),
    "inputs": {k: file_fingerprint(p) for k, p in {
        "tracks": TRACKS_PATH, "fluo": FLUO_PATH, "cell_class": CELL_CLASS_PATH,
        "mask_features": MASK_FEATURES_PATH, "masks": MASK_PATH, "raw_metadata": RAW_METADATA_PATH}.items()},
    "r_um_definition": R_UM_DEFINITION,
    "units": {"positions": "µm (z, y, x)", "speed": "µm/min", "time": TIME_LABEL,
              "scale_vec_zyx_um": list(scale_vec), "frame_interval_sec": FRAME_INTERVAL_SEC,
              "units_confirmed_from_raw": RAW_METADATA_PATH is not None},
    "config": config_snapshot(), "packages": package_versions(), "run_log": RUN_LOG,
}

export = df_all.drop(columns=[c for c in ("qc_bright_lo", "qc_bright_hi", "qc_vol_lo", "qc_vol_hi") if c in df_all])
for c in export.select_dtypes("float64").columns:
    export[c] = export[c].astype(np.float32)
for c in ("qc_reason", "qc_primary", "dist_bin", "movie_segment", "embryo_id", "end_reason", f"{SECOND_CH}_status",
          f"{SECOND_CH}_dist_bin_at_first_positive"):
    if c in export:
        export[c] = export[c].astype("category")
nuc_path = write_table(export, OUT_DIR / f"{DATASET_NAME}_nuclei_{EXPORT_VERSION}_{stamp}.parquet", provenance)
trk_path = write_table(trk.reset_index(), OUT_DIR / f"{DATASET_NAME}_tracks_{EXPORT_VERSION}_{stamp}.parquet", provenance)
(OUT_DIR / f"{DATASET_NAME}_params_{EXPORT_VERSION}_{stamp}.json").write_text(json.dumps(_jsonable(provenance), indent=2))

# per-embryo summary row (#10)
ok = df_all[df_all["qc_good_frame"] & df_all["track_ok"]]
row = {"embryo_id": EMBRYO_ID, "dataset": DATASET_NAME, "export": nuc_path.name,
       "n_frames": int(df_all["t"].nunique()), f"start_{TIME_LABEL}": float(df_all["hours"].min()),
       f"end_{TIME_LABEL}": float(df_all["hours"].max()), "n_tracks": int(len(trk)),
       "n_tracks_ok": int(trk["track_ok"].sum()), "n_nucleus_frames": int(len(df_all)),
       "qc_pass_frac": float(df_all["qc_good_frame"].mean()),
       **{f"qc_fail_{r}_frac": float(df_all[f"qc_bad_{r}"].mean()) for r in QC_REASONS},
       "center": R_UM_DEFINITION, "reporter": SECOND_CH, "units_confirmed": RAW_METADATA_PATH is not None}
for seg, g in ok.groupby("movie_segment"):
    row[f"{seg}_median_speed_um_min"] = float(g["speed_um_min"].median())
    row[f"{seg}_median_persistence"] = float(g["persistence"].median())
    if STATUS_READY:
        row[f"{seg}_{SECOND_CH}_positive_frac"] = float(g[f"{SECOND_CH}_positive"].mean())
if STATUS_READY:
    row.update({"threshold_mode": THRESHOLD_MODE, "fixed_threshold": FIXED_THRESHOLD, "min_consecutive": MIN_CONSECUTIVE_FRAMES,
                f"{SECOND_CH}_tracks_ever_positive": int(trk[f"{SECOND_CH}_track_ever_positive"].sum()),
                f"{SECOND_CH}_median_onset_{TIME_LABEL}": float(trk.loc[trk[f"{SECOND_CH}_onset_observed"], f"{SECOND_CH}_first_positive_hours"].median())})
summ_path = OUT_DIR.parent / "embryo_summary.csv"
emb = pd.read_csv(summ_path) if summ_path.exists() else pd.DataFrame()
if len(emb):
    emb = emb[emb["dataset"] != DATASET_NAME]
emb = pd.concat([emb, pd.DataFrame([row])], ignore_index=True)
emb.to_csv(summ_path, index=False)
print("wrote:", nuc_path.name, "|", trk_path.name, "| embryo summary rows:", len(emb))
display(pd.Series(row).to_frame("this embryo"))

## 14. Extended QC: track number and length

**Question:** is tracking quality stable over time and equal for positive and negative nuclei?

- Turnover = (track starts + ends) / nuclei per frame, excluding the first/last frame and divisions. High turnover marks phases where tracks break.
- Track length by status: if ever-positive tracks are systematically longer (bright nuclei are easier to follow), any track-level comparison between positive and negative is biased toward long-lived positives (survivorship).

In [ ]:
pf = df_all.groupby("t").agg(n_all=("track_id", "size"), n_deep=("qc_bad_not_deep", lambda s: (~s).sum()),
                            n_qc_pass=("qc_good_frame", "sum"))
starts = trk[(trk["t_start"] > T_FIRST) & ~trk["starts_from_division"]].groupby("t_start").size()
ends = trk[(trk["t_end"] < T_LAST) & ~trk["ends_in_division"]].groupby("t_end").size()
divs = trk[trk["ends_in_division"]].groupby("t_end").size()
pf["track_starts"] = starts.reindex(pf.index).fillna(0)
pf["track_ends"] = ends.reindex(pf.index).fillna(0)
pf["divisions"] = divs.reindex(pf.index).fillna(0)
pf["turnover"] = (pf["track_starts"] + pf["track_ends"]) / pf["n_all"]
if STATUS_READY:
    pp = df_all[df_all["qc_good_frame"]].groupby(["t", f"{SECOND_CH}_status"]).size().unstack(fill_value=0)
    pf = pf.join(pp.add_prefix("n_"))
hh = frame_to_hours(pf.index)
rows_ = [["n_all", "n_deep", "n_qc_pass"] + ([f"n_positive", f"n_negative"] if STATUS_READY else []),
         ["track_starts", "track_ends", "divisions"], ["turnover"]]
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, subplot_titles=("nuclei per frame", "track events per frame", "turnover"))
for i, cols in enumerate(rows_, start=1):
    for j, c in enumerate(cols):
        if c in pf:
            y = pf[c].rolling(5, center=True, min_periods=1).mean() if i > 1 else pf[c]
            colr = {"n_positive": STATUS_COLORS["positive"], "n_negative": STATUS_COLORS["negative"]}.get(c, SEGMENT_COLORS[j % 5])
            fig.add_trace(go.Scatter(x=hh, y=y, name=c, line=dict(color=colr)), i, 1)
fig.update_xaxes(title_text=TIME_LABEL, row=3, col=1)
fig = apply_dark_theme(fig, 13).update_layout(height=750, title="Tracking QC over time (events: 5-frame rolling mean)")
save_fig(fig, "14_tracking_qc"); fig.show()

if STATUS_READY:
    _t = trk.copy()
    _t["status"] = np.where(_t[f"{SECOND_CH}_track_ever_positive"], "ever positive", "never positive")
    fig = px.histogram(_t, x=np.log10(_t["n_obs"]), color="status", barmode="overlay", histnorm="probability density",
                       nbins=60, opacity=0.5, color_discrete_map={"ever positive": STATUS_COLORS["positive"], "never positive": STATUS_COLORS["negative"]},
                       labels={"x": "log10 track length (frames)"}, title="Track length by status")
    apply_dark_theme(fig, 14).show()
    print(_t.groupby("status")["n_obs"].describe()[["count", "25%", "50%", "75%"]])
log_step("14_tracking_qc", median_turnover=float(pf["turnover"].median()),
         max_turnover_hours=float(frame_to_hours(pf["turnover"].idxmax())))
pf.to_csv(OUT_DIR / "per_frame_qc.csv")

## 15. Masks with tails

Inputs: `t` (frame) and `tail` (frames of history). Masks (label MIP) and tails are coloured by second-channel status (`STATUS_LEVEL`: per-frame status at `t`, or "track" = ever positive); QC-failing nuclei are grey. Use `crop=(y0, y1, x0, x1)` in mask pixels to zoom.

The first call for a new frame builds and caches its label MIP (tens of seconds, ~2.7 GB peak).

In [ ]:
def plot_masks_with_tails(t, tail=TAIL_LENGTH_FRAMES, crop=VIEW_CROP_YX, status_level=STATUS_LEVEL, show_mip=True):
    mip_lab = label_mip(t)
    sub = df_all[df_all["t"] == t]
    if STATUS_READY:
        st = (np.where(sub[f"{SECOND_CH}_track_ever_positive"], "positive", "negative") if status_level == "track"
              else sub[f"{SECOND_CH}_status"].to_numpy())
    else:
        st = np.full(len(sub), "none")
    st = np.where(sub["qc_good_frame"], st, "qc_fail")
    status_of = dict(zip(sub["track_id"].to_numpy(), st))
    names = list(STATUS_COLORS)
    lut = np.zeros(int(max(mip_lab.max(), sub["track_id"].max())) + 1, dtype=np.int8)
    lut[sub["track_id"].to_numpy()] = [names.index(s) + 1 for s in st]
    code = lut[mip_lab]
    rgba = np.zeros(mip_lab.shape + (4,), np.float32)
    for i, s in enumerate(names, start=1):
        rgba[code == i] = mcolors.to_rgba(STATUS_COLORS[s], alpha=0.45)
    bnd = find_boundaries(mip_lab, mode="inner")
    rgba[bnd & (code > 0), 3] = 0.95

    fig, ax = plt.subplots(figsize=(9, 9))
    if show_mip:
        img = read_mip_frame(NUCLEAR_CH, t)
        ax.imshow(img, cmap="gray", clim=contrast_limits(img), extent=mip_on_mask_grid_extent())
    ax.imshow(rgba, interpolation="nearest")
    hist = df_all[(df_all["t"] > t - tail) & (df_all["t"] <= t) & df_all["track_id"].isin(status_of)]
    for s in names:
        segs = [g[["cx_px", "cy_px"]].to_numpy() for tid, g in hist.sort_values("t").groupby("track_id")
                if status_of.get(tid) == s and len(g) > 1]
        if segs:
            ax.add_collection(LineCollection(segs, colors=STATUS_COLORS[s], linewidths=1.0, alpha=0.9))
    counts = pd.Series(st).value_counts()
    ax.legend(handles=[Line2D([], [], color=STATUS_COLORS[s], lw=3, label=f"{s} (n={counts.get(s, 0)})") for s in names],
              loc="upper right", fontsize=8, framealpha=0.3)
    ax.set_xlim(0, X_MASK); ax.set_ylim(Y_MASK, 0)
    crop_view(ax, crop, t)
    ax.set_title(f"t={t} ({float(frame_to_hours(t)):.2f} {TIME_LABEL}), tails {tail} frames "
                 f"({tail * DT_MIN:.0f} min), status: {status_level}")
    ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    return fig

fig = plot_masks_with_tails(VIEW_T, TAIL_LENGTH_FRAMES)
save_fig(fig, f"15_masks_tails_t{VIEW_T}_tail{TAIL_LENGTH_FRAMES}")
plt.show()

## 16. Run log

Every section above appended what it ran with which settings; this writes it next to the exports.

In [ ]:
(OUT_DIR / f"run_log_{stamp}.json").write_text(json.dumps(_jsonable({"config": config_snapshot(), "steps": RUN_LOG}), indent=2))
display(pd.DataFrame([{"section": r["section"], "time": r["time"]} for r in RUN_LOG]))
print("run log:", OUT_DIR / f"run_log_{stamp}.json")